# Faster R-CNN · Kaggle FT / 3e-4 · AMP

**학습 조건:** COCO_V1 → 전체 미세조정(FT), AdamW, LR 3e-4, 이미지 640,
**실제 batch 4 × 기울기 누적 4회 = 목표 실효 batch 16**, 최대 100 epoch, patience 30, AMP FP16, seed **0·1·2**.
각 seed는 COCO에서 독립적으로 시작한다. 기존 FP32 seed 0은 새 AMP 실험에 섞지 않는다.
W&B 이름 예: `cj_faster_e100_s1_fix_amp_acc4_ft_lr3e-4`.

**실행 순서:** 환경·내장 함수 → 셀 2 설정 → 셀 3 데이터 → 셀 4 누적 갱신 확인 → **셀 5 본 학습** → 셀 6~10 분석·공유.
셀 11~12는 선택 사항이다. 학습 셀은 최대 100 epoch를 실행하므로 분석할 때는 셀 5를 건너뛴다.
별도의 `RUN_CANDIDATES=True` 스위치 없이 **셀 5 실행이 학습 시작**이다.

**현재 Kaggle 준비 상태:** `Pill Team Python 3.12` 커널과 팀 `uv.lock` 설치를 완료했고,
Drive 데이터는 `/kaggle/working/pill-training-data/data`에 복사한다.
환경 검사와 학습 셀은 데이터 다운로드·설치·커널 재시작을 하지 않는다.
새 VM을 만들면 환경과 데이터를 다시 준비해야 한다. 같은 VM에서는 커널만 다시 연결해도 다운로드한 데이터가 남아 있다.
다운로드 중이라면 끝난 뒤 셀 2부터 진행한다. `.part` 파일이 남아 있으면 데이터 준비를 중단한다.

**평가:** bbox mAP@[0.75:0.05:0.95], max detections=100, 원본 category ID·픽셀 좌표.
검증 추론과 validation loss는 FP32이다. 검증 loss는 고정 batch 1·측정 seed 0으로 epoch마다
별도 forward하며 가중치·정규화 버퍼·학습 난수 상태를 보존한다.
train loss는 실제 batch 4·AMP의 이미지 수 가중 평균이다. 마지막 묶음은 남은 이미지 수로 정규화한다.
일반 묶음은 16개마다, 마지막 묶음은 16개 미만이어도 한 번 갱신한다.
검출 loss의 내부 정규화·RoI 샘플링·연산 정밀도 때문에 물리 batch 16과 완전히 같은 결과를 보장하지 않는다.
train/val loss의 절댓값은 측정 조건을 함께 보고 해석한다.
best와 early stopping의 기준은 validation mAP이다.

**출력:** `/kaggle/working/runs/cj_faster_experiments`, `/kaggle/working/submissions/cj_faster_experiments`.
결과 보관은 셀 12의 ZIP을 다운로드하거나 Kaggle 출력으로 별도 저장한다.
현재 추가된 환경 복구·다운로드 셀은 데이터가 없을 때만 사용한다. 기존 9개 후보 탐색은 수행하지 않는다.
공통 함수는 모두 이 노트북 안에 있으며 별도 소스 모듈 업로드는 필요 없다.

[PyTorch AMP·기울기 누적](https://docs.pytorch.org/docs/2.14/notes/amp_examples.html#gradient-accumulation) ·
[torchvision Faster R-CNN](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html)


## 1. 환경·공통 함수


### 셀 1 · 현재 Python·팀 잠금 버전·GPU 확인

설치가 완료된 `Pill Team Python 3.12` 커널에서 실행한다. 버전을 바꾸거나 패키지를 설치하지 않는다.
Kaggle 기본 Python 3.13 커널이면 3.12 커널로 먼저 전환한다.


In [ ]:
from pathlib import Path
import subprocess

python = Path("/kaggle/working/pill-object-detection/.venv/bin/python")
data = Path("/kaggle/working/pill-training-data/data")
runs = Path("/kaggle/working/runs/cj_faster_experiments")

print("기존 Python 환경:", python.is_file())
print("학습 이미지 폴더:", (data / "raw/train_images").is_dir())
print("분할 CSV:", (data / "processed/split.csv").is_file())
print("저장된 last.pt 개수:", sum(1 for _ in runs.rglob("last.pt")))

if python.is_file():
    subprocess.run(
        [
            str(python), "-m", "ipykernel", "install",
            "--user",
            "--name", "pill-team-py312",
            "--display-name", "Pill Team Python 3.12",
        ],
        check=True,
    )
    print("Pill 커널 재등록 완료. 커널 목록을 새로고침하세요.")
else:
    print("기존 환경이 없습니다. 환경 재설치가 필요합니다.")

In [ ]:
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

if not Path("/kaggle/working").is_dir():
    raise RuntimeError(
        "현재 Kaggle 서버가 아닙니다. Kaggle의 VS Code Compatible URL로 다시 연결하세요."
    )

if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU를 찾지 못했습니다. Kaggle의 GPU 설정을 확인하세요.")

# uv 준비
uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
if not Path(uv).is_file():
    script = urllib.request.urlopen(
        "https://astral.sh/uv/install.sh"
    ).read().decode()
    subprocess.run(["sh"], input=script, text=True, check=True)

# 팀 저장소 준비
repo = Path("/kaggle/working/pill-object-detection")
if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", "exp/23-faster-rcnn-baseline",
            "https://github.com/codeit-ai-15th/pill-object-detection.git",
            str(repo),
        ],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        check=True,
    )

# 팀 잠금 버전으로 Python 3.12 환경 복구
subprocess.run(
    [uv, "sync", "--locked", "--python", "3.12"],
    cwd=repo,
    env={**os.environ, "UV_PROJECT_ENVIRONMENT": str(repo / ".venv")},
    check=True,
)

# 커널 등록
subprocess.run(
    [
        str(repo / ".venv/bin/python"),
        "-m", "ipykernel", "install",
        "--user",
        "--name", "pill-team-py312",
        "--display-name", "Pill Team Python 3.12",
    ],
    check=True,
)

print("복구 완료: 커널 목록을 새로고침하고 Pill Team Python 3.12를 선택하세요.")

In [ ]:
import json
import shutil
import urllib.request
import urllib.parse
import urllib.error
from pathlib import Path
from getpass import getpass

TOKEN = getpass("Google Drive Access token: ").strip()
DEST = Path("/kaggle/working/pill-training-data/data")
ROOT_ID = "1PDgLX8HnXcp9vT73Ot1ngvTjDtjnhxEL"
FOLDER_TYPE = "application/vnd.google-apps.folder"
counts = {"downloaded": 0, "existing": 0}


def children(folder_id):
    items, page = [], None
    while True:
        params = {
            "q": f"'{folder_id}' in parents and trashed = false",
            "fields": "nextPageToken,files(id,name,mimeType,size)",
            "pageSize": 1000,
        }
        if page:
            params["pageToken"] = page
        request = urllib.request.Request(
            "https://www.googleapis.com/drive/v3/files?"
            + urllib.parse.urlencode(params),
            headers={"Authorization": f"Bearer {TOKEN}"},
        )
        with urllib.request.urlopen(request, timeout=60) as response:
            result = json.load(response)
        items.extend(result.get("files", []))
        page = result.get("nextPageToken")
        if not page:
            return items


def download(item, parent):
    name = item["name"]
    if name.startswith("."):
        return
    if Path(name).name != name:
        raise RuntimeError(f"파일 이름을 확인하세요: {name}")

    target = parent / name
    if item["mimeType"] == FOLDER_TYPE:
        target.mkdir(parents=True, exist_ok=True)
        for child in children(item["id"]):
            download(child, target)
        return

    if item["mimeType"].startswith("application/vnd.google-apps."):
        raise RuntimeError(f"일반 데이터 파일이 아닙니다: {name}")

    size = int(item["size"]) if "size" in item else None
    if size is not None and target.is_file() and target.stat().st_size == size:
        counts["existing"] += 1
        return

    parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(
        f"https://www.googleapis.com/drive/v3/files/{item['id']}?alt=media",
        headers={"Authorization": f"Bearer {TOKEN}"},
    )
    temporary = target.with_name(target.name + ".part")
    with urllib.request.urlopen(request, timeout=120) as response:
        with temporary.open("wb") as output:
            shutil.copyfileobj(response, output, length=1024 * 1024)

    if size is not None and temporary.stat().st_size != size:
        raise RuntimeError(f"파일 크기가 다릅니다. 이 셀을 다시 실행하세요: {name}")

    temporary.replace(target)
    counts["downloaded"] += 1
    if counts["downloaded"] % 10 == 0:
        print(f"다운로드 {counts['downloaded']}개 완료", flush=True)


try:
    folders = {item["name"]: item for item in children(ROOT_ID)}
    for section, names in {
        "raw": ("train_images", "train_annotations"),
        "processed": ("split.csv", "label_fixes.csv"),
    }.items():
        items = {
            item["name"]: item
            for item in children(folders[section]["id"])
        }
        for name in names:
            print(f"{section}/{name} 복사 중…", flush=True)
            download(items[name], DEST / section)

    print("복사 완료:", counts)
    print("저장 위치:", DEST)
except urllib.error.HTTPError as error:
    raise RuntimeError(
        f"Drive 요청 실패: HTTP {error.code}. "
        "401이면 새 Access token으로 이 셀을 다시 실행하세요."
    ) from None
finally:
    del TOKEN

In [ ]:
import sys
import os
import json
import hashlib
import importlib.metadata as metadata
import shutil
import subprocess
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("팀 기준은 Python 3.12입니다. 오른쪽 위에서 Pill Team Python 3.12 커널을 선택하세요.")
import tomllib

IS_KAGGLE = Path("/kaggle/working").is_dir()
REPO_ROOT_OVERRIDE = None
if REPO_ROOT_OVERRIDE is not None:
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()
elif IS_KAGGLE:
    REPO_ROOT = Path("/kaggle/working/pill-object-detection")
else:
    REPO_ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
                      if (path / "pyproject.toml").is_file() and (path / "uv.lock").is_file()), None)
if REPO_ROOT is None or not REPO_ROOT.is_dir():
    raise RuntimeError("팀 저장소가 없습니다. Kaggle 환경 설치 때 사용한 저장소를 준비하고 REPO_ROOT_OVERRIDE를 지정하세요.")

reference_names = (".python-version", "pyproject.toml", "uv.lock")
missing = [name for name in reference_names if not (REPO_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"팀 환경 기준 파일이 없습니다: {missing}. 환경 설치에 사용한 저장소 경로를 확인하세요.")
python_pin = tuple(int(part) for part in (REPO_ROOT / ".python-version").read_text().strip().split("."))
if tuple(sys.version_info[:len(python_pin)]) != python_pin:
    raise RuntimeError("저장소의 Python 기준과 다릅니다. 팀 Python 3.12 커널을 선택하세요.")
ENV_REFERENCE_SHA256 = {name: hashlib.sha256((REPO_ROOT / name).read_bytes()).hexdigest()
                        for name in reference_names}
TEAM_LOCK_SHA256 = "be9c6c31eb9b5c836ca010112de5a2e7bbdf72a2119d8ac937c8dc9d3a36fea6"
if ENV_REFERENCE_SHA256["uv.lock"] != TEAM_LOCK_SHA256:
    raise RuntimeError("학습 준비 때 확인한 팀 uv.lock과 다릅니다. 버전을 자동 변경하지 말고 사용할 팀 잠금 파일을 확인하세요.")
uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
if not Path(uv).is_file():
    raise RuntimeError("uv를 찾지 못했습니다. 환경 설치 때 사용한 uv 경로를 확인하세요.")
checked = subprocess.run(
    [uv, "sync", "--locked", "--check", "--python", sys.executable],
    cwd=REPO_ROOT, env={**os.environ, "UV_PROJECT_ENVIRONMENT": sys.prefix},
    capture_output=True, text=True,
)
if checked.returncode:
    raise RuntimeError("현재 커널이 팀 잠금 환경과 다릅니다. 저장소에서 uv sync --locked --python 3.12를 완료한 뒤 "
                       "해당 .venv 커널을 다시 선택하세요.\n" + checked.stderr[-3000:])

import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from torchvision.ops import nms

for package, module_name in {"numpy": "numpy", "pandas": "pandas", "pillow": "PIL",
                            "matplotlib": "matplotlib", "torch": "torch", "torchvision": "torchvision",
                            "torchmetrics": "torchmetrics", "wandb": "wandb"}.items():
    module = sys.modules.get(module_name)
    loaded = getattr(module, "__version__", None)
    if loaded is not None and loaded != metadata.version(package):
        raise RuntimeError(f"{package}의 메모리 버전과 설치 버전이 다릅니다. 같은 VM의 커널만 재시작하고 셀 1부터 실행하세요.")
if IS_KAGGLE and not torch.cuda.is_available():
    raise RuntimeError("Kaggle CUDA GPU가 없습니다. 현재 서버의 GPU 설정과 선택한 커널을 확인하세요.")
if torch.cuda.is_available():
    boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.]], device="cuda")
    scores = torch.tensor([0.9, 0.8], device="cuda")
    keep = nms(boxes, scores, 0.5)
    torch.cuda.synchronize()
    if keep.cpu().tolist() != [0]:
        raise RuntimeError("torchvision GPU NMS 결과가 다릅니다. CUDA 빌드와 GPU 환경을 확인하세요.")
    del boxes, scores, keep
    print("CUDA·torchvision GPU NMS: 통과")

commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True)
ENV_REPOSITORY_COMMIT = commit.stdout.strip() if commit.returncode == 0 else "unavailable"
ENVIRONMENT = {
    "python": sys.version.split()[0], "python_executable": sys.executable,
    "packages": {dist.metadata["Name"].lower().replace("_", "-"): dist.version
                 for dist in metadata.distributions() if dist.metadata.get("Name")},
    "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "uv_lock_sha256": ENV_REFERENCE_SHA256["uv.lock"],
    "environment_repository_commit": ENV_REPOSITORY_COMMIT,
}
print("Python:", ENVIRONMENT["python"], "| torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("GPU:", ENVIRONMENT["device"], "| CUDA:", ENVIRONMENT["cuda"])
print("현재 커널과 팀 uv.lock 일치: 확인 완료. 설치·다운로드·재시작 없음.")


### 내장 함수 A · bbox·시각화·코드 식별자

위에서부터 한 번씩 실행한다. 이 셀은 함수를 정의하며 학습을 시작하지 않는다.


In [ ]:
# 이 셀부터 함수 구현을 노트북 안에 직접 정의한다. 별도 프로젝트 모듈을 읽지 않는다.
import ast
import hashlib
import inspect
import json
import linecache
import textwrap
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

NOTEBOOK_CODE_BLOCKS = {}


def _register_notebook_block(name):
    """실행한 셀의 소스를 기록하여 재개 검증과 Cascade 작업 전달에 사용한다.

    Args:
        name (str): 이 노트북 안의 코드 묶음 이름.
    """
    frame = inspect.currentframe().f_back
    try:
        source = "".join(linecache.getlines(frame.f_code.co_filename))
        if not source:
            shell = get_ipython()
            source = shell.history_manager.input_hist_raw[-1]
        if not source.strip():
            raise RuntimeError("실행 셀의 소스를 찾을 수 없습니다. Jupyter/Colab에서 내장 함수 셀부터 다시 실행하세요.")
        NOTEBOOK_CODE_BLOCKS[name] = source
    finally:
        del frame


def _notebook_source_hash(names):
    """셀 번호·주석을 제외한 실제 내장 코드의 식별자를 계산한다.

    Args:
        names (list): 해시할 코드 묶음 이름.

    Returns:
        str: Python 3.11과 3.12에서 공통으로 비교할 수 있는 소스 해시.
    """
    entries = {}
    for name in names:
        if name not in NOTEBOOK_CODE_BLOCKS:
            raise RuntimeError(f"내장 함수 셀이 빠졌습니다: {name}. 위에서부터 함수 셀을 실행하세요.")
        tree = ast.parse(NOTEBOOK_CODE_BLOCKS[name])
        for node in ast.walk(tree):
            # 이 코드에서는 제네릭 타입 문법을 쓰지 않으므로 3.12의 빈 필드만 제외한다.
            if hasattr(node, "type_params") and not node.type_params:
                del node.type_params
        entries[name] = ast.dump(tree, include_attributes=False)
    return hashlib.sha256(json.dumps(entries, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def common_code_identity():
    """세 노트북에 같은 내용으로 들어간 데이터·평가 함수의 해시를 반환한다.

    Returns:
        str: 비교에 사용할 공통 소스 식별자.
    """
    return _notebook_source_hash(["helpers", "data", "analysis"])


def xyxy_to_xywh(bbox):
    """Ultralytics 추론 결과의 xyxy bbox를 xywh로 바꾼다.

    Args:
        bbox (list[float]): [좌상단 x, 좌상단 y, 우하단 x, 우하단 y]. 원본 픽셀.

    Returns:
        list[float]: [좌상단 x, 좌상단 y, 너비, 높이]. 원본 픽셀.
    """
    x1, y1, x2, y2 = bbox
    return [x1, y1, x2 - x1, y2 - y1]

def draw_boxes(image, boxes, labels=None, color="red", width=4):
    """이미지에 xywh bbox를 그려 새 PIL 이미지로 돌려준다.

    기본 글꼴은 한글을 그리지 못하므로 labels에는 category_id 같은 숫자·영문을 넣는다.

    Args:
        image (str | Path | PIL.Image.Image): 이미지 경로나 PIL 이미지.
        boxes (list[list[float]]): 그릴 bbox 목록. 각각 [좌상단 x, 좌상단 y, 너비, 높이], 원본 픽셀.
        labels (list | None): bbox마다 위에 적을 글자. boxes와 순서가 같다. None이면 적지 않는다.
        color (str): 선과 글자의 색.
        width (int): 선 두께(픽셀).

    Returns:
        PIL.Image.Image: bbox를 그린 RGB 이미지.
    """
    image = Image.open(image) if not isinstance(image, Image.Image) else image
    image = image.convert("RGB")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default(size=28)
    for i, (x, y, w, h) in enumerate(boxes):
        draw.rectangle([x, y, x + w, y + h], outline=color, width=width)
        if labels:
            draw.text((x, max(0, y - 32)), str(labels[i]), fill=color, font=font)
    return image

_register_notebook_block("helpers")


### 내장 함수 B · 데이터·고정 분할·라벨 수정

위에서부터 한 번씩 실행한다. 이 셀은 함수를 정의하며 학습을 시작하지 않는다.


In [ ]:
"""알약 검출의 고정 분할·수정 라벨·원본 좌표를 관리한다."""

import ast
import hashlib
import json
import os
import shutil
import tempfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision.ops import box_convert
from torchvision.transforms.functional import to_tensor


def file_sha256(path):
    """파일 내용을 해시한다.

    Args:
        path (str | Path): 검사할 파일.

    Returns:
        str: SHA-256.
    """
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def stable_hash(value):
    """JSON으로 표현 가능한 설정의 식별자를 만든다.

    Args:
        value (object): 해시할 값.

    Returns:
        str: SHA-256.
    """
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     allow_nan=False).encode()).hexdigest()


def write_json(path, value):
    """임시 파일을 완성한 뒤 원자적으로 JSON을 저장한다.

    Args:
        path (str | Path): 저장 위치.
        value (object): JSON 값.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=path.name, delete=False) as stream:
        temporary = Path(stream.name)
        stream.write(text)
    os.replace(temporary, path)


def _bbox(value):
    if isinstance(value, str):
        value = ast.literal_eval(value)
    array = np.asarray(value, dtype=float)
    if array.shape != (4,) or not np.isfinite(array).all() or (array[2:] <= 0).any():
        raise ValueError("bbox는 유한한 xywh 숫자 4개와 양수 너비·높이여야 합니다. 라벨표를 수정하세요.")
    return array.tolist()


def prepare_data(image_dir, annotation_dir, split_path, fixes_path):
    """원본은 보존하고 고정 분할과 add/modify 수정표를 적용한다.

    Args:
        image_dir (str | Path): 학습 이미지 폴더.
        annotation_dir (str | Path): COCO JSON 폴더.
        split_path (str | Path): file_name, combo, split 열의 고정 분할.
        fixes_path (str | Path): file_name, category_id, action, bbox 열의 수정표.

    Returns:
        dict: 이미지 목록·클래스 매핑·객체 라벨·데이터 식별자.
    """
    image_dir, annotation_dir = Path(image_dir), Path(annotation_dir)
    split_path, fixes_path = Path(split_path), Path(fixes_path)
    for path in (image_dir, annotation_dir, split_path, fixes_path):
        if not path.exists():
            raise FileNotFoundError(f"데이터 경로가 없습니다: {path}. 노트북 1번의 경로를 지정하세요.")
    json_paths = sorted(annotation_dir.rglob("*.json"))
    if not json_paths:
        raise ValueError("annotation JSON이 없습니다. train_annotations 경로를 확인하세요.")
    images, annotations, names, source_hashes = {}, [], {}, []
    for path in json_paths:
        doc = json.loads(path.read_text(encoding="utf-8-sig"))
        local_images = {int(item["id"]): item for item in doc["images"]}
        for item in local_images.values():
            name = item["file_name"]
            if Path(name).name != name:
                raise ValueError("이미지 파일명에 경로가 포함됐습니다. 이미지 manifest를 확인하세요.")
            record = {"file_name": name, "image_id": int(item["id"]),
                      "width": int(item["width"]), "height": int(item["height"])}
            if name in images and images[name] != record:
                raise ValueError(f"같은 이미지의 ID·크기가 다릅니다: {name}. 원본 연결을 확인하세요.")
            images[name] = record
        for category in doc.get("categories", []):
            names[int(category["id"])] = str(category.get("name") or category["id"])
        for ann in doc.get("annotations", []):
            if int(ann["image_id"]) not in local_images:
                raise ValueError(f"JSON 내부 image_id가 연결되지 않습니다: {path}. 라벨을 확인하세요.")
            annotations.append({"file_name": local_images[int(ann["image_id"])]["file_name"],
                                "category_id": int(ann["category_id"]), "bbox": _bbox(ann["bbox"]),
                                "iscrowd": int(ann.get("iscrowd", 0))})
        source_hashes.append((str(path.relative_to(annotation_dir)), file_sha256(path)))
    if not annotations:
        raise ValueError("객체 라벨이 없습니다. annotation 내용을 확인하세요.")
    raw_count = len(annotations)
    fixes = pd.read_csv(fixes_path, keep_default_na=False)
    if "bbox" not in fixes.columns and "bbox_after" in fixes.columns:
        fixes = fixes.rename(columns={"bbox_after": "bbox"})
    required = {"file_name", "category_id", "action", "bbox"}
    if not required.issubset(fixes.columns):
        raise ValueError(f"수정표에 {sorted(required)} 열이 필요합니다. label_fixes.csv를 확인하세요.")
    changed = Counter()
    for row in fixes.to_dict("records"):
        name, category = str(row["file_name"]).strip(), int(row["category_id"])
        if name not in images:
            raise ValueError(f"수정할 이미지가 원본에 없습니다: {name}. 수정표를 확인하세요.")
        bbox, action = _bbox(row["bbox"]), str(row["action"]).strip().lower()
        # 팀 공유표의 bbox·area도 기존 객체 수정으로 정규화한다.
        action = {"bbox": "modify", "area": "modify"}.get(action, action)
        matches = [a for a in annotations if a["file_name"] == name and a["category_id"] == category]
        if action == "add":
            if any(np.allclose(a["bbox"], bbox, atol=1e-6, rtol=0) for a in matches):
                raise ValueError(f"추가 라벨이 중복됩니다: {name}, {category}. 수정표를 확인하세요.")
            annotations.append({"file_name": name, "category_id": category, "bbox": bbox, "iscrowd": 0})
        elif action == "modify":
            if len(matches) != 1:
                raise ValueError(f"수정 대상이 {len(matches)}개입니다: {name}, {category}. 라벨을 구분하세요.")
            matches[0]["bbox"] = bbox
        else:
            raise ValueError(f"지원하지 않는 수정 action: {action}. add·modify·bbox·area로 지정하세요.")
        changed[action] += 1
    frame = pd.DataFrame(sorted(images.values(), key=lambda r: r["file_name"]))
    if frame["image_id"].duplicated().any():
        raise ValueError("서로 다른 이미지에 같은 image_id가 있습니다. 원본 연결을 확인하세요.")
    split = pd.read_csv(split_path, dtype=str, keep_default_na=False)
    if not {"file_name", "combo", "split"}.issubset(split.columns):
        raise ValueError("분할표에 file_name·combo·split 열이 필요합니다. 고정 분할표를 확인하세요.")
    if split["file_name"].duplicated().any() or set(split["file_name"]) != set(frame["file_name"]):
        raise ValueError("분할표의 이미지 목록이 원본과 다릅니다. 누락·중복을 확인하세요.")
    if set(split["split"]) != {"train", "val"}:
        raise ValueError("분할에는 train·val이 모두 있어야 합니다. 고정 분할표를 확인하세요.")
    if not split["combo"].eq(split["file_name"].str.split("_").str[0]).all():
        raise ValueError("분할표의 combo와 파일명 조합이 다릅니다. 고정 분할표를 확인하세요.")
    if split.groupby("combo")["split"].nunique().max() > 1:
        raise ValueError("같은 조합이 train·val에 섞였습니다. 팀의 고정 분할표를 확인하세요.")
    frame = frame.merge(split, on="file_name", validate="one_to_one")
    image_hashes = []
    for row in frame.to_dict("records"):
        path = image_dir / row["file_name"]
        with Image.open(path) as image:
            if image.size != (row["width"], row["height"]):
                raise ValueError(f"이미지와 라벨의 크기가 다릅니다: {path}. 좌표 기준을 확인하세요.")
        image_hashes.append((row["file_name"], file_sha256(path)))
    hash_splits = defaultdict(set)
    for name, digest in image_hashes:
        hash_splits[digest].add(split.set_index("file_name").loc[name, "split"])
    if any(len(parts) > 1 for parts in hash_splits.values()):
        raise ValueError("동일한 이미지 내용이 train·val에 중복됩니다. 고정 분할을 검토하세요.")
    categories = sorted({a["category_id"] for a in annotations})
    mapping = {category: index + 1 for index, category in enumerate(categories)}
    objects = defaultdict(list)
    for ann in annotations:
        info = images[ann["file_name"]]
        x, y, w, h = ann["bbox"]
        if x < 0 or y < 0 or x + w > info["width"] or y + h > info["height"]:
            raise ValueError(f"경계 밖 bbox: {ann['file_name']}, {ann['category_id']}. 수정표를 확인하세요.")
        if ann["iscrowd"] != 0:
            raise ValueError("이 실험은 개별 알약 iscrowd=0을 전제로 합니다. 평가 정책을 확인하세요.")
        xyxy = box_convert(torch.tensor([ann["bbox"]], dtype=torch.float64), "xywh", "xyxy")[0].tolist()
        objects[ann["file_name"]].append({"boxes": xyxy, "category_id": ann["category_id"],
                                          "label": mapping[ann["category_id"]]})
    class_map = pd.DataFrame({"category_id": categories,
                              "model_label": [mapping[c] for c in categories],
                              "name": [names.get(c, str(c)) for c in categories]})
    identity = stable_hash({"images": image_hashes, "annotations": source_hashes,
                            "split": file_sha256(split_path), "fixes": file_sha256(fixes_path),
                            "mapping": mapping})
    return {"frame": frame, "objects": dict(objects), "class_map": class_map,
            "category_to_label": mapping, "label_to_category": {v: k for k, v in mapping.items()},
            "image_dir": image_dir, "identity": identity, "image_hashes": dict(image_hashes),
            "raw_boxes": raw_count, "fixed_boxes": len(annotations), "fixes": dict(changed)}


def data_summary(bundle):
    """분할별 이미지 수와 클래스별 정답 객체 수를 집계한다.

    Args:
        bundle (dict): prepare_data 결과.

    Returns:
        tuple: 분할 요약표, 클래스별 객체 수.
    """
    rows = []
    for row in bundle["frame"].to_dict("records"):
        for obj in bundle["objects"].get(row["file_name"], []):
            rows.append({"split": row["split"], "category_id": obj["category_id"]})
    counts = pd.crosstab(pd.Series([r["category_id"] for r in rows], name="category_id"),
                         pd.Series([r["split"] for r in rows], name="split"))
    counts = counts.reindex(index=bundle["class_map"]["category_id"], columns=["train", "val"], fill_value=0)
    summary = pd.DataFrame([{"split": part, "images": int(bundle["frame"]["split"].eq(part).sum()),
                             "boxes": int(counts[part].sum()), "classes": int((counts[part] > 0).sum())}
                            for part in ("train", "val")])
    return summary, counts.reset_index()


def cache_images(bundle, cache_root):
    """내용 해시가 일치하는 로컬 이미지 캐시를 준비한다.

    Args:
        bundle (dict): 데이터 묶음.
        cache_root (str | Path): 복사본 저장 루트.

    Returns:
        dict: image_dir만 캐시 경로로 바꾼 묶음.
    """
    destination = Path(cache_root) / bundle["identity"][:12]
    destination.mkdir(parents=True, exist_ok=True)
    for name, digest in bundle["image_hashes"].items():
        target = destination / name
        if not target.exists() or file_sha256(target) != digest:
            with tempfile.NamedTemporaryFile(dir=destination, delete=False) as stream:
                temporary = Path(stream.name)
            shutil.copyfile(bundle["image_dir"] / name, temporary)
            if file_sha256(temporary) != digest:
                raise IOError("이미지 복사본이 원본과 다릅니다. 캐시를 다시 준비하세요.")
            os.replace(temporary, target)
    return {**bundle, "image_dir": destination}


class PillDataset(Dataset):
    """원본 픽셀 이미지와 torchvision용 정답을 반환한다."""

    def __init__(self, frame, image_dir, objects=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.objects = objects

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.image_dir / row["file_name"]) as image:
            tensor = to_tensor(image.convert("RGB"))
        objects = [] if self.objects is None else self.objects.get(row["file_name"], [])
        boxes = torch.tensor([o["boxes"] for o in objects], dtype=torch.float32).reshape(-1, 4)
        return tensor, {"boxes": boxes, "labels": torch.tensor([o["label"] for o in objects], dtype=torch.int64),
                        "image_id": torch.tensor(int(row["image_id"])),
                        "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
                        "iscrowd": torch.zeros(len(objects), dtype=torch.int64)}


def collate_detection(batch):
    """객체 수가 다른 이미지와 정답을 목록으로 묶는다.

    Args:
        batch (list): 이미지·정답 쌍.

    Returns:
        tuple: 이미지 목록, 정답 목록.
    """
    return tuple(zip(*batch))


def make_dataset(bundle, split):
    """지정된 고정 분할의 Dataset을 만든다.

    Args:
        bundle (dict): 데이터 묶음.
        split (str): train 또는 val.

    Returns:
        PillDataset: 선택 분할.
    """
    return PillDataset(bundle["frame"].loc[bundle["frame"]["split"].eq(split)],
                       bundle["image_dir"], bundle["objects"])


def cache_dataset(dataset, cache_root):
    """Test를 포함한 Dataset의 이미지를 검증된 로컬 캐시로 연결한다.

    Args:
        dataset (PillDataset): 복사할 이미지 목록.
        cache_root (str | Path): 캐시 루트.

    Returns:
        PillDataset: 같은 순서·정답을 유지하는 로컬 Dataset.
    """
    hashes = {name: file_sha256(dataset.image_dir / name) for name in dataset.frame["file_name"]}
    source = {"image_dir": dataset.image_dir, "image_hashes": hashes,
              "identity": stable_hash({"files": hashes, "frame": dataset.frame.to_dict("records")})}
    cached = cache_images(source, cache_root)
    return PillDataset(dataset.frame, cached["image_dir"], dataset.objects)


def test_dataset(image_dir):
    """숫자 파일명을 대회 image_id로 보존한 Test Dataset을 만든다.

    Args:
        image_dir (str | Path): Test 이미지 폴더.

    Returns:
        PillDataset: 정답이 없는 이미지 목록.
    """
    image_dir = Path(image_dir)
    rows = []
    for path in sorted(image_dir.iterdir()):
        if path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
            continue
        if not path.stem.isdigit():
            raise ValueError(f"Test 파일명이 숫자가 아닙니다: {path.name}. 대회 파일을 확인하세요.")
        with Image.open(path) as image:
            width, height = image.size
        rows.append({"file_name": path.name, "image_id": int(path.stem), "width": width, "height": height})
    frame = pd.DataFrame(rows)
    if frame.empty or frame["image_id"].duplicated().any():
        raise ValueError("Test 이미지가 없거나 image_id가 중복됩니다. 경로·파일명을 확인하세요.")
    return PillDataset(frame.sort_values("image_id"), image_dir)


_register_notebook_block("data")


### 내장 함수 C · mAP·혼동행렬·오류·CSV

위에서부터 한 번씩 실행한다. 이 셀은 함수를 정의하며 학습을 시작하지 않는다.


In [ ]:
"""모델에 독립적인 원본 좌표 평가·오류 분석·제출 변환."""

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou, box_convert




IOU_THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
SUBMISSION_COLUMNS = ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y",
                      "bbox_w", "bbox_h", "score"]


def evaluate_records(records, class_map=None, detailed=True):
    """공통 예측의 bbox mAP와 선택적 클래스·IoU별 AP를 계산한다.

    Args:
        records (list): prediction과 target을 포함한 이미지별 결과. labels는 원래 category ID.
        class_map (DataFrame | None): 전체 클래스 목록.
        detailed (bool): 상세 결과 계산 여부.

    Returns:
        dict: 요약, 클래스별 AP, IoU별 AP, PR 곡선 원자료.
    """
    if not records or any("target" not in row for row in records):
        raise ValueError("정답이 있는 validation 결과가 필요합니다. 4번 검증 추론을 실행하세요.")
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", iou_thresholds=IOU_THRESHOLDS,
                                  max_detection_thresholds=[1, 10, 100], class_metrics=detailed,
                                  extended_summary=detailed)
    metric.update([row["prediction"] for row in records], [row["target"] for row in records])
    result = metric.compute()
    summary = {"map_75_95": float(result["map"]), "map_75": float(result["map_75"]),
               "mar_100": float(result["mar_100"]), "images": len(records),
               "ground_truths": sum(len(r["target"]["boxes"]) for r in records)}
    if not detailed:
        return {"summary": summary}
    counts = Counter(int(c) for row in records for c in row["target"]["labels"])
    precision = result["precision"][:, :, :, 0, -1].numpy()
    classes = result["classes"].reshape(-1).tolist()

    def valid_mean(values):
        valid = values[values >= 0]
        return float(valid.mean()) if valid.size else None

    ap = pd.DataFrame([{"category_id": int(category), "ap_75_95": valid_mean(precision[:, :, index]),
                        "val_objects": counts[int(category)]} for index, category in enumerate(classes)])
    if class_map is not None:
        ap = class_map[["category_id", "name"]].merge(ap, how="left", on="category_id")
        ap["val_objects"] = ap["val_objects"].fillna(0).astype(int)
    ap.loc[ap["val_objects"].eq(0), "ap_75_95"] = np.nan
    iou_ap = pd.DataFrame({"iou": IOU_THRESHOLDS,
                           "ap": [valid_mean(precision[index]) for index in range(len(IOU_THRESHOLDS))]})
    return {"summary": summary, "classes": ap, "ious": iou_ap,
            "precision": precision, "category_ids": classes}


def detection_errors(records, categories, iou_threshold=0.75, score_threshold=0.30):
    """클래스와 독립적인 IoU 일대일 대응으로 혼동과 미검출을 집계한다.

    Args:
        records (list): validation 공통 결과.
        categories (list): 원래 category ID 목록.
        iou_threshold (float): 대응 최소 IoU.
        score_threshold (float): 진단용 최소 score.

    Returns:
        tuple: 행=정답·열=예측 혼동행렬, 오류 상세 표.
    """
    if not 0 < iou_threshold <= 1 or not 0 <= score_threshold <= 1:
        raise ValueError("진단 IoU·score 범위가 잘못됐습니다. 4번 설정을 확인하세요.")
    categories = [int(c) for c in categories]
    positions = {category: index for index, category in enumerate(categories)}
    background = len(categories)
    matrix = np.zeros((background + 1, background + 1), dtype=int)
    events = []
    for record in records:
        target, pred = record["target"], record["prediction"]
        kept = torch.where(pred["scores"] >= score_threshold)[0]
        boxes, labels, scores = pred["boxes"][kept], pred["labels"][kept], pred["scores"][kept]
        overlaps = box_iou(target["boxes"], boxes).numpy()
        candidates = np.argwhere(overlaps >= iou_threshold)
        candidates = sorted(candidates.tolist(), key=lambda pair: (-overlaps[tuple(pair)], pair))
        used_gt, used_pred = set(), set()

        def append(kind, gt=None, prediction=None, overlap=None):
            events.append({"image_id": int(record["image_id"]), "file_name": record["file_name"],
                           "kind": kind, "gt_index": gt,
                           "pred_index": None if prediction is None else int(kept[prediction]),
                           "true_category": None if gt is None else int(target["labels"][gt]),
                           "pred_category": None if prediction is None else int(labels[prediction]),
                           "score": None if prediction is None else float(scores[prediction]),
                           "iou": overlap})

        for gt, prediction in candidates:
            if gt in used_gt or prediction in used_pred:
                continue
            used_gt.add(gt)
            used_pred.add(prediction)
            truth, estimate = int(target["labels"][gt]), int(labels[prediction])
            matrix[positions[truth], positions[estimate]] += 1
            append("correct" if truth == estimate else "class_confusion", gt, prediction,
                   float(overlaps[gt, prediction]))
        for gt in range(len(target["boxes"])):
            if gt not in used_gt:
                matrix[positions[int(target["labels"][gt])], background] += 1
                possible = [p for p in range(len(boxes)) if int(labels[p]) == int(target["labels"][gt])
                            and 0 < overlaps[gt, p] < iou_threshold]
                best = max(possible, key=lambda p: overlaps[gt, p]) if possible else None
                append("localization_or_missed" if best is not None else "missed", gt, best,
                       None if best is None else float(overlaps[gt, best]))
        for prediction in range(len(boxes)):
            if prediction not in used_pred:
                matrix[background, positions[int(labels[prediction])]] += 1
                duplicate = any(int(target["labels"][gt]) == int(labels[prediction])
                                and overlaps[gt, prediction] >= iou_threshold for gt in used_gt)
                append("duplicate" if duplicate else "extra", prediction=prediction)
    labels = [str(c) for c in categories] + ["unmatched"]
    columns = ["image_id", "file_name", "kind", "gt_index", "pred_index", "true_category",
               "pred_category", "score", "iou"]
    return pd.DataFrame(matrix, index=labels, columns=labels), pd.DataFrame(events, columns=columns)


def plot_ground_truth(dataset, reverse, count=3):
    """입력 데이터의 정답 bbox를 표시한다.

    Args:
        dataset (PillDataset): 정답이 있는 이미지.
        reverse (dict): 모델 번호에서 원래 클래스 ID로 가는 매핑.
        count (int): 표시할 이미지 수.

    Returns:
        Figure: 정답 박스 이미지.
    """
    fig, axes = plt.subplots(1, min(count, len(dataset)), figsize=(6 * min(count, len(dataset)), 6), squeeze=False)
    for index, axis in enumerate(axes[0]):
        _, target = dataset[index]
        image = draw_boxes(dataset.image_dir / dataset.frame.iloc[index]["file_name"],
                           [xyxy_to_xywh(box.tolist()) for box in target["boxes"]],
                           [str(reverse[int(c)]) for c in target["labels"]], color="lime")
        axis.imshow(image)
        axis.set(title=dataset.frame.iloc[index]["file_name"][:30])
        axis.axis("off")
    fig.tight_layout()
    return fig


def plot_learning_curves(run_dirs):
    """완료 여부와 관계없이 저장된 학습 곡선을 표시한다.

    Args:
        run_dirs (list): metrics.csv를 포함한 실행 폴더.

    Returns:
        Figure: loss·mAP 곡선.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for directory in run_dirs:
        path = Path(directory) / "metrics.csv"
        if not path.exists():
            continue
        history = pd.read_csv(path)
        if history.empty:
            continue
        name = Path(directory).name
        axes[0].plot(history["epoch"], history["train_loss"], label=name)
        axes[1].plot(history["epoch"], history["val_map_75_95"], label=name)
    for axis, title in zip(axes, ["Train loss", "Validation mAP@[.75:.95]"]):
        axis.set(title=title, xlabel="Epoch")
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend(fontsize=7)
    fig.tight_layout()
    return fig


def plot_candidate_scores(results):
    """설정별 seed 점수와 평균·표본 표준편차를 함께 표시한다.

    Args:
        results (DataFrame): 완료된 실행 결과.

    Returns:
        Figure: seed별 점수와 설정별 평균. 단일 seed의 표준편차는 추정하지 않음.
    """
    summary = results.groupby("candidate_id")["best_val_map_75_95"].agg(["mean", "std", "count"])
    summary = summary.sort_values("mean", ascending=False)
    positions = {key: index for index, key in enumerate(summary.index)}
    x = np.arange(len(summary))
    fig, axis = plt.subplots(figsize=(max(8, len(summary)*1.2), 5))
    axis.bar(x, summary["mean"], color="lightsteelblue", label="Mean")
    estimated = summary["count"].ge(2).to_numpy()
    axis.errorbar(x[estimated], summary.loc[estimated, "mean"], yerr=summary.loc[estimated, "std"],
                  fmt="none", ecolor="black", capsize=4, label="Sample SD (n>=2)")
    for seed, group in results.groupby("seed"):
        axis.scatter([positions[key] for key in group["candidate_id"]], group["best_val_map_75_95"],
                     label=f"seed {int(seed)}", zorder=3)
    axis.set(xticks=x, xticklabels=[f"{key}\nn={int(summary.loc[key, 'count'])}" for key in summary.index],
             ylabel="Validation mAP@[.75:.95]", ylim=(0, 1.05),
             title="Candidate and seed comparison (n=1: SD unavailable)")
    axis.tick_params(axis="x", rotation=30, labelsize=8)
    axis.legend(fontsize=8)
    axis.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    return fig


def plot_ap(report):
    """낮은 클래스 AP·정답 수와 IoU별 AP를 함께 표시한다.

    Args:
        report (dict): 상세 evaluate_records 결과.

    Returns:
        Figure: 클래스·IoU별 그래프.
    """
    low = report["classes"].dropna(subset=["ap_75_95"]).nsmallest(15, "ap_75_95")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh([f"{int(row.category_id)} (n={int(row.val_objects)})" for row in low.itertuples()], low["ap_75_95"])
    axes[0].invert_yaxis()
    axes[0].set(title="Lowest class AP (validation support)", xlabel="AP@[.75:.95]", xlim=(0, 1))
    axes[1].plot(report["ious"]["iou"], report["ious"]["ap"], marker="o")
    axes[1].set(title="AP by IoU", xlabel="IoU", ylabel="AP", ylim=(0, 1))
    axes[1].grid(alpha=0.25)
    fig.tight_layout()
    return fig


def plot_confusion(matrix, normalize=True):
    """정답 행 기준 정규화 또는 원시 건수 혼동행렬을 표시한다.

    Args:
        matrix (DataFrame): 행=정답·열=예측. unmatched 행은 추가 예측, 열은 미검출.
        normalize (bool): 정답 행별 비율 표시 여부.

    Returns:
        Figure: 진단 혼동행렬.
    """
    values = matrix.to_numpy(dtype=float)
    if normalize:
        values = np.divide(values, values.sum(axis=1, keepdims=True),
                           out=np.zeros_like(values), where=values.sum(axis=1, keepdims=True) > 0)
    size = max(9, min(20, len(matrix) * 0.28))
    fig, axis = plt.subplots(figsize=(size, size))
    heatmap = axis.imshow(values, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    axis.set(xticks=range(len(matrix)), yticks=range(len(matrix)),
             xticklabels=matrix.columns, yticklabels=matrix.index,
             xlabel="Predicted (unmatched = missed)", ylabel="True (unmatched = extra)",
             title="Detection confusion: row-normalized" if normalize else "Detection confusion: counts")
    axis.tick_params(axis="x", rotation=90, labelsize=7)
    axis.tick_params(axis="y", labelsize=7)
    fig.colorbar(heatmap, ax=axis, fraction=0.04)
    fig.tight_layout()
    return fig


def preview_records(records, events=None, count=6, score_threshold=0.30):
    """정답·예측과 오류 종류를 표시한다. Test에는 정답을 그리지 않는다.

    Args:
        records (list): 공통 예측 결과.
        events (DataFrame | None): validation 오류 표.
        count (int): 최대 이미지 수.
        score_threshold (float): 표시용 점수 하한.

    Returns:
        Figure: 초록 정답·빨강 예측 이미지.
    """
    if events is not None:
        errors = events.loc[events["kind"].ne("correct")].copy()
        errors = errors.sort_values("score", ascending=False, na_position="last")
        ids = list(dict.fromkeys(errors["image_id"].tolist()))[:count]
        chosen = [r for identifier in ids for r in records if r["image_id"] == identifier]
    else:
        chosen = records[:count]
    if not chosen:
        fig, axis = plt.subplots(figsize=(8, 2))
        axis.text(0.5, 0.5, "No error images at the chosen diagnostic thresholds", ha="center")
        axis.axis("off")
        return fig
    fig, axes = plt.subplots(len(chosen), 1, figsize=(9, 6 * len(chosen)), squeeze=False)
    for record, axis in zip(chosen, axes[:, 0]):
        with Image.open(record["image_path"]) as source:
            image = source.convert("RGB")
        if "target" in record:
            target = record["target"]
            image = draw_boxes(image, [xyxy_to_xywh(b.tolist()) for b in target["boxes"]],
                               [f"GT:{int(c)}" for c in target["labels"]], color="lime")
        pred = record["prediction"]
        indices = torch.where(pred["scores"] >= score_threshold)[0]
        indices = indices[torch.argsort(pred["scores"][indices], descending=True)][:20]
        captions = []
        for index in indices:
            caption = f"P:{int(pred['labels'][index])} {float(pred['scores'][index]):.2f}"
            if "target" in record and len(record["target"]["boxes"]):
                overlap = float(box_iou(pred["boxes"][index].reshape(1, 4), record["target"]["boxes"]).max())
                caption += f" I:{overlap:.2f}"
            captions.append(caption)
        image = draw_boxes(image, [xyxy_to_xywh(pred["boxes"][i].tolist()) for i in indices],
                           captions, color="red")
        image.thumbnail((1280, 1280))
        kinds = "prediction only; no ground truth"
        if events is not None:
            kinds = ", ".join(sorted(set(events.loc[events["image_id"].eq(record["image_id"]), "kind"])))
        axis.imshow(image)
        axis.set_title(f"ID {record['image_id']} | {kinds}\ngreen=GT, red=prediction; score>={score_threshold}", fontsize=9)
        axis.axis("off")
    fig.tight_layout()
    return fig


def submission_frame(records, allowed_categories):
    """공통 원본 좌표 예측을 대회 CSV 표로 변환한다.

    Args:
        records (list): labels가 원래 category ID인 공통 예측.
        allowed_categories (list): 허용 클래스 목록.

    Returns:
        DataFrame: 대회 컬럼 순서의 표. 빈 예측도 빈 표로 표현한다.
    """
    allowed_categories = set(int(c) for c in allowed_categories)
    if len({int(r["image_id"]) for r in records}) != len(records):
        raise ValueError("예측 이미지 ID가 중복됩니다. 전체 추론 목록을 확인하세요.")
    rows = []
    for record in records:
        pred = record["prediction"]
        if not (len(pred["boxes"]) == len(pred["labels"]) == len(pred["scores"])):
            raise ValueError("예측 boxes·labels·scores 수가 다릅니다. 공통 변환을 확인하세요.")
        for box, category, score in zip(pred["boxes"], pred["labels"], pred["scores"]):
            x, y, w, h = xyxy_to_xywh(box.tolist())
            if int(category) not in allowed_categories:
                raise ValueError("알 수 없는 category ID입니다. 모델의 클래스 매핑을 확인하세요.")
            if not np.isfinite([x, y, w, h, float(score)]).all() or not 0 <= float(score) <= 1:
                raise ValueError("유한하지 않은 좌표·score가 있습니다. 모델 출력을 확인하세요.")
            if x < -1e-4 or y < -1e-4 or w <= 0 or h <= 0 or x+w > record["width"]+1e-4 or y+h > record["height"]+1e-4:
                raise ValueError("원본 경계 밖 또는 면적 0 예측입니다. 좌표 복원을 확인하세요.")
            rows.append({"image_id": int(record["image_id"]), "category_id": int(category),
                         "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h, "score": float(score)})
    frame = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS[1:])
    frame = frame.sort_values(["image_id", "score"], ascending=[True, False], kind="stable").reset_index(drop=True)
    frame.insert(0, "annotation_id", np.arange(1, len(frame)+1, dtype=np.int64))
    for column in ("annotation_id", "image_id", "category_id"):
        frame[column] = frame[column].astype("int64")
    return frame[SUBMISSION_COLUMNS]


def restore_submission(frame, records):
    """CSV를 다시 읽은 뒤 이미지별 공통 예측으로 복원한다.

    Args:
        frame (DataFrame): 제출 컬럼의 표.
        records (list): 전체 이미지·정답 목록.

    Returns:
        list: 빈 이미지도 보존한 복원 결과.
    """
    if list(frame.columns) != SUBMISSION_COLUMNS or frame.isna().any().any():
        raise ValueError("제출 컬럼·결측값이 잘못됐습니다. CSV 생성 코드를 확인하세요.")
    try:
        frame = frame.apply(pd.to_numeric, errors="raise")
    except (TypeError, ValueError) as error:
        raise ValueError("CSV에 숫자가 아닌 값이 있습니다. 제출 파일을 다시 생성하세요.") from error
    if not np.isfinite(frame.to_numpy(dtype=float)).all():
        raise ValueError("CSV에 유한하지 않은 값이 있습니다. 제출 파일을 다시 생성하세요.")
    for column in ("annotation_id", "image_id", "category_id"):
        if not np.equal(frame[column], np.floor(frame[column])).all():
            raise ValueError("CSV ID는 정수여야 합니다. 클래스·이미지 매핑을 확인하세요.")
    if frame["annotation_id"].duplicated().any() or not set(frame["image_id"]).issubset({r["image_id"] for r in records}):
        raise ValueError("제출 ID가 중복되거나 이미지 목록과 다릅니다. CSV를 확인하세요.")
    restored = []
    for record in records:
        rows = frame.loc[frame["image_id"].eq(record["image_id"])]
        boxes = torch.tensor(rows[["bbox_x", "bbox_y", "bbox_w", "bbox_h"]].to_numpy(dtype=np.float32), dtype=torch.float32).reshape(-1, 4)
        restored.append({**record, "prediction": {
            "boxes": box_convert(boxes, "xywh", "xyxy"),
            "labels": torch.tensor(rows["category_id"].to_numpy(dtype=np.int64), dtype=torch.int64),
            "scores": torch.tensor(rows["score"].to_numpy(dtype=np.float32), dtype=torch.float32)}})
    return restored


def verify_roundtrip(records, categories, csv_path):
    """실제 CSV 저장·읽기와 전후 mAP·좌표·ID 일치를 검사한다.

    Args:
        records (list): 정답을 포함한 validation 예측.
        categories (list): 원래 클래스 ID 목록.
        csv_path (str | Path): 검증용 CSV 저장 위치.

    Returns:
        dict: 변환 전후 mAP와 통과 결과.
    """
    path = Path(csv_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = submission_frame(records, categories)
    frame.to_csv(path, index=False)
    loaded = pd.read_csv(path)
    restored = restore_submission(loaded, records)
    for before, after in zip(records, restored):
        left, right = before["prediction"], after["prediction"]
        order = torch.argsort(left["scores"], descending=True, stable=True)
        if not torch.equal(left["labels"][order], right["labels"]) or not torch.allclose(left["boxes"][order], right["boxes"], atol=1e-3, rtol=0) or not torch.allclose(left["scores"][order], right["scores"], atol=1e-7, rtol=0):
            raise ValueError("CSV 복원 전후 예측이 바뀌었습니다. ID·좌표 변환을 확인하세요.")
    before = evaluate_records(records, detailed=False)["summary"]["map_75_95"]
    after = evaluate_records(restored, detailed=False)["summary"]["map_75_95"]
    if abs(before-after) > 1e-6:
        raise ValueError("CSV 변환 전후 mAP가 다릅니다. 평가 입력과 좌표 변환을 확인하세요.")
    report = {"before": before, "after": after, "passed": True, "images": len(records), "rows": len(frame)}
    write_json(path.with_suffix(".json"), report)
    return report


_register_notebook_block("analysis")


### 내장 함수 D · AMP·기울기 누적·검증 loss·체크포인트

위에서부터 한 번씩 실행한다. 이 셀은 함수를 정의하며 학습을 시작하지 않는다.


In [ ]:
"""Kaggle에서 FT / 3e-4 독립 seed 학습과 AMP·검증·체크포인트를 관리한다."""

import contextlib
import gc
import importlib.metadata
import json
import math
import os
import random
import sys
import tempfile
import time
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import FrozenBatchNorm2d




DEFAULT_CONFIG = {
    "max_epochs": 100, "patience": 30, "image_size": 640, "batch_size": 4, "accumulation_steps": 4,
    "weight_decay": 1e-4, "betas": [0.9, 0.999], "warmup_epochs": 3,
    "final_lr_ratio": 0.01, "num_workers": 0, "score_threshold": 0.001,
    "nms_iou": 0.5, "max_detections": 100, "preview_score": 0.30,
    "preview_count": 3, "experiment_tag": "fix_amp_acc4", "wandb_project": "pill-object-detection",
    "wandb_entity": None, "wandb_mode": "online", "use_wandb": True,
    "require_cuda": True, "deterministic": True, "precision": "amp_fp16",
    "normalization": "FrozenBatchNorm2d_eps0_fixed_statistics_and_affine",
    "augmentation": "none", "weights": "COCO_V1", "optimizer": "AdamW",
    "learning_rate": 3e-4, "learning_rates": [3e-4],
    "val_loss_batch_size": 1, "loss_measurement_seed": 0,
    "inference_precision": "fp32", "val_loss_precision": "fp32",
}
MODES = ("ft",)


def runtime_info():
    """실제로 import한 학습 환경을 기록한다.

    Returns:
        dict: 패키지·CUDA·GPU 정보.
    """
    versions = {name: importlib.metadata.version(name) for name in
                ("torch", "torchvision", "torchmetrics", "numpy", "pandas", "wandb", "pycocotools", "pillow", "matplotlib")}
    return {"packages": versions, "python": sys.version.split()[0], "cuda": torch.version.cuda,
            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "framework": "torchvision", "model": "faster_rcnn_r50_fpn",
            "uv_lock_sha256": globals().get("ENV_REFERENCE_SHA256", {}).get("uv.lock")}


def code_identity():
    """별도 Python 파일 없이 실제 실행한 내장 함수의 해시를 반환한다.

    Returns:
        dict: 공통 코드와 해당 모델 학습 코드의 소스 식별자.
    """
    names = ["helpers", "data", "analysis", "training"]
    if "cascade_model" in NOTEBOOK_CODE_BLOCKS:
        names.append("cascade_model")
    return {name: _notebook_source_hash([name]) for name in names}


def validate_config(config):
    """지원하는 설정과 수치 범위를 검사한다.

    Args:
        config (dict): DEFAULT_CONFIG에 기반한 실험 설정.
    """
    if set(config) != set(DEFAULT_CONFIG):
        raise ValueError("설정 키가 기본 규격과 다릅니다. DEFAULT_CONFIG를 복사해 수정하세요.")
    for key in ("max_epochs", "patience", "image_size", "batch_size", "accumulation_steps", "preview_count"):
        if type(config[key]) is not int or config[key] <= 0:
            raise ValueError(f"{key}는 양의 정수여야 합니다. 1번 설정을 확인하세요.")
    if config["weights"] != "COCO_V1" or config["optimizer"] != "AdamW":
        raise ValueError("COCO_V1·AdamW만 지원합니다. 셀 2의 설정을 확인하세요.")
    if config["precision"] not in {"fp32", "amp_fp16"}:
        raise ValueError("precision은 fp32 또는 amp_fp16으로 지정하세요.")
    if config["inference_precision"] != "fp32" or config["val_loss_precision"] != "fp32":
        raise ValueError("검증 추론과 validation loss 측정은 FP32입니다. 셀 2의 설정을 확인하세요.")
    if config["augmentation"] != "none" or config["normalization"] != DEFAULT_CONFIG["normalization"]:
        raise ValueError("증강·정규화 변경은 별도 구현이 필요합니다. 지원하는 공통 정책을 사용하세요.")
    if not 0 <= config["warmup_epochs"] < config["max_epochs"] or not 0 < config["final_lr_ratio"] <= 1:
        raise ValueError("warmup·최종 학습률 비율이 잘못됐습니다. 1번 설정을 확인하세요.")
    if not 0 <= config["score_threshold"] <= 1 or not 0 < config["nms_iou"] <= 1 or config["max_detections"] != 100:
        raise ValueError("score·NMS 또는 평가 max detections가 잘못됐습니다. 최대 100개 정책을 확인하세요.")
    if config["weight_decay"] < 0 or len(config["betas"]) != 2 or any(not 0 <= v < 1 for v in config["betas"]):
        raise ValueError("AdamW 설정이 잘못됐습니다. decay·betas를 확인하세요.")
    if config["num_workers"] < 0 or config["wandb_mode"] not in {"online", "offline", "disabled"}:
        raise ValueError("workers 또는 W&B mode가 잘못됐습니다. 1번 설정을 확인하세요.")
    if not config["experiment_tag"] or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in config["experiment_tag"]):
        raise ValueError("실험 태그는 영문·숫자·밑줄·하이픈으로 지정하세요.")
    if config["learning_rate"] != 3e-4:
        raise ValueError("이번 실험은 FT / 3e-4로 고정합니다. 셀 2의 learning_rate를 확인하세요.")
    if config["learning_rates"] != [config["learning_rate"]]:
        raise ValueError("비교 자료의 learning_rates는 실제 단일 학습률과 같아야 합니다.")
    if not isinstance(config["val_loss_batch_size"], int) or config["val_loss_batch_size"] < 1:
        raise ValueError("val_loss_batch_size는 양의 정수로 지정하세요.")
    if not isinstance(config["loss_measurement_seed"], int) or config["loss_measurement_seed"] < 0:
        raise ValueError("loss_measurement_seed는 0 이상의 정수로 지정하세요.")


def seed_everything(seed, deterministic=True):
    """분할을 변경하지 않고 학습 난수만 설정한다.

    Args:
        seed (int): 학습 seed.
        deterministic (bool): 결정적 연산 사용 여부.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)


def lr_text(lr):
    """학습률을 읽기 쉬운 이름으로 표현한다.

    Args:
        lr (float): 학습률.

    Returns:
        str: 예를 들어 3e-4.
    """
    mantissa, exponent = f"{lr:.8e}".split("e")
    return f"{mantissa.rstrip('0').rstrip('.')}e{int(exponent)}"


def experiment_spec(mode, lr, seed):
    """학습 범위·학습률·seed로 실험을 식별한다.

    Args:
        mode (str): 전체 미세조정 ft.
        lr (float): 학습률.
        seed (int): 0, 1, 2.

    Returns:
        dict: 후보 ID와 실행 조건.
    """
    if mode not in MODES or not math.isfinite(lr) or lr <= 0 or seed not in (0, 1, 2):
        raise ValueError("실험은 ft, 양수 학습률, seed 0·1·2로 지정하세요.")
    candidate = f"{mode}_lr{lr_text(lr)}"
    return {"candidate_id": candidate, "spec_id": f"{candidate}_s{seed}",
            "mode": mode, "lr": float(lr), "seed": int(seed)}





def run_name(config, spec):
    """사용자 지정 접두부와 조건을 포함한 W&B 이름을 만든다.

    Args:
        config (dict): 공통 설정.
        spec (dict): 실험 조건.

    Returns:
        str: cj_faster_e100_s1_fix_ft_lr3e-4 형식.
    """
    return (f"cj_faster_e{config['max_epochs']}_s{spec['seed']}_{config['experiment_tag']}_"
            f"{spec['mode']}_lr{lr_text(spec['lr'])}")


def _recipe(config, bundle):
    validate_config(config)
    return {"config": config, "data_identity": bundle["identity"], "code": code_identity(),
            "runtime": runtime_info(), "common_code": common_code_identity(), "class_mapping": {str(k): v for k, v in bundle["category_to_label"].items()}}


def experiment_root(root, config, bundle):
    """이번 설계의 코드·환경·데이터·설정과 일치하는 결과만 모은다.

    Args:
        root (str | Path): 결과 루트.
        config (dict): 공통 설정.
        bundle (dict): 데이터 묶음.

    Returns:
        Path: 실험 묶음 경로.
    """
    return Path(root) / stable_hash(_recipe(config, bundle))[:12]


def _frozen_normalization(module):
    for name, child in list(module.named_children()):
        if isinstance(child, torch.nn.BatchNorm2d):
            frozen = FrozenBatchNorm2d(child.num_features, eps=0.0)
            for attribute in ("weight", "bias", "running_mean", "running_var"):
                getattr(frozen, attribute).copy_(getattr(child, attribute).detach())
            setattr(module, name, frozen)
        elif isinstance(child, FrozenBatchNorm2d):
            child.eps = 0.0
        else:
            _frozen_normalization(child)


def build_model(num_classes, spec, config, load_coco=True):
    """같은 COCO 검출 모델에서 출력층과 학습 범위를 설정한다.

    Args:
        num_classes (int): 배경을 제외한 클래스 수.
        spec (dict): 실험 조건.
        config (dict): 공통 설정.
        load_coco (bool): 신규 학습은 True. 저장된 전체 체크포인트 복원 시에만 False.

    Returns:
        Module: CPU의 Faster R-CNN. 모든 정규화 통계·계수는 고정.
    """
    validate_config(config)
    experiment_spec(spec["mode"], spec["lr"], spec["seed"])
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1 if load_coco else None
    model = fasterrcnn_resnet50_fpn(weights=weights, weights_backbone=None,
                                    trainable_backbone_layers=5 if load_coco else None,
                                    min_size=config["image_size"], max_size=config["image_size"])
    _frozen_normalization(model)
    features = model.roi_heads.box_predictor.cls_score.in_features
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(spec["seed"])
        model.roi_heads.box_predictor = FastRCNNPredictor(features, num_classes + 1)
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    model.roi_heads.score_thresh = config["score_threshold"]
    model.roi_heads.nms_thresh = config["nms_iou"]
    model.roi_heads.detections_per_img = config["max_detections"]
    return model


def parameter_summary(model):
    """실제 학습 모듈과 파라미터 수를 확인한다.

    Args:
        model (Module): 검사 모델.

    Returns:
        DataFrame: 모듈별 전체·학습 가능한 파라미터 수.
    """
    groups = {}
    for name, parameter in model.named_parameters():
        group = ".".join(name.split(".")[:3]) if name.startswith("backbone.body") else name.split(".")[0]
        row = groups.setdefault(group, {"module": group, "total": 0, "trainable": 0})
        row["total"] += parameter.numel()
        row["trainable"] += parameter.numel() if parameter.requires_grad else 0
    return pd.DataFrame(groups.values())


def make_optimizer(model, spec, config):
    """고정 파라미터를 제외하고 AdamW decay 그룹을 만든다.

    Args:
        model (Module): 학습 모델.
        spec (dict): 학습률 조건.
        config (dict): AdamW 설정.

    Returns:
        AdamW: bias·1차원 파라미터에는 decay 미적용.
    """
    groups = [[], []]
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            groups[int(name.endswith("bias") or parameter.ndim <= 1)].append(parameter)
    return torch.optim.AdamW([{"params": groups[0], "weight_decay": config["weight_decay"]},
                               {"params": groups[1], "weight_decay": 0.0}],
                              lr=spec["lr"], betas=tuple(config["betas"]))


def make_scheduler(optimizer, steps_per_epoch, config):
    """optimizer step 단위의 선형 warmup·cosine 스케줄러를 만든다.

    Args:
        optimizer (Optimizer): AdamW.
        steps_per_epoch (int): 누적 후 epoch당 예정 optimizer 갱신 수.
        config (dict): 학습 예산·warmup 설정.

    Returns:
        LambdaLR: 첫 step부터 warmup을 적용하는 scheduler.
    """
    total = config["max_epochs"] * steps_per_epoch
    warmup = int(config["warmup_epochs"] * steps_per_epoch)

    def factor(step):
        if warmup and step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, max(0.0, (step-warmup) / max(1, total-warmup-1)))
        return config["final_lr_ratio"] + (1-config["final_lr_ratio"]) * (1+math.cos(math.pi*progress))/2

    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def _device(config):
    if config["require_cuda"] and not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 없습니다. Kaggle GPU의 Pill Team Python 3.12 커널을 선택하세요.")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _loader(dataset, config, training=False, generator=None):
    return DataLoader(dataset, batch_size=config["batch_size"] if training else 1,
                      shuffle=training, num_workers=config["num_workers"],
                      collate_fn=collate_detection, generator=generator,
                      pin_memory=torch.cuda.is_available())


def _synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def collect_predictions(model, dataset, config, reverse, with_targets=True):
    """추론을 한 번 수행해 원본 category ID의 공통 결과를 만든다.

    Args:
        model (Module): 복원 또는 학습 모델.
        dataset (PillDataset): 순서가 고정된 이미지 목록.
        config (dict): 추론 설정.
        reverse (dict): 모델 번호→원래 category ID.
        with_targets (bool): validation 정답 포함 여부.

    Returns:
        tuple: 공통 결과, 순수 추론 시간·전체 시간.
    """
    device = next(model.parameters()).device
    records, model_seconds = [], 0.0
    start_all = time.perf_counter()
    model.eval()
    with torch.inference_mode():
        for images, targets in _loader(dataset, config):
            inputs = [image.to(device) for image in images]
            _synchronize(device)
            start = time.perf_counter()
            outputs = model(inputs)
            _synchronize(device)
            model_seconds += time.perf_counter()-start
            for output, target in zip(outputs, targets):
                info = dataset.frame.iloc[len(records)].to_dict()
                prediction = {key: output[key].detach().cpu() for key in ("boxes", "labels", "scores")}
                prediction["labels"] = torch.tensor([reverse[int(c)] for c in prediction["labels"]], dtype=torch.int64)
                record = {key: int(info[key]) for key in ("image_id", "width", "height")}
                record.update(file_name=info["file_name"], image_path=str(dataset.image_dir / info["file_name"]),
                              prediction=prediction)
                if with_targets:
                    record["target"] = {"boxes": target["boxes"].cpu(),
                                        "labels": torch.tensor([reverse[int(c)] for c in target["labels"]], dtype=torch.int64)}
                records.append(record)
    return records, {"model_ms_per_image": 1000*model_seconds/max(1, len(records)),
                     "wall_seconds": time.perf_counter()-start_all, "images": len(records)}


def smoke_test(bundle, config, spec):
    """본 학습과 같은 작은 배치·누적·정밀도로 한 번의 갱신을 확인한다.

    Args:
        bundle (dict): 고정 학습·검증 데이터.
        config (dict): 본 학습과 동일한 설정.
        spec (dict): 독립 학습 조건.

    Returns:
        dict: 실제 optimizer 갱신·고정 파라미터·시각화·CSV 확인 결과.
    """
    import matplotlib.pyplot as plt
    device = _device(config)
    seed_everything(spec["seed"], config["deterministic"])
    dataset = make_dataset(bundle, "train")
    if len(dataset) < config["batch_size"] * config["accumulation_steps"]:
        raise ValueError("학습 이미지가 목표 실효 batch보다 적습니다. 셀 2의 batch·누적 횟수와 분할을 확인하세요.")
    model = optimizer = scaler = before = None
    try:
        model = build_model(len(bundle["class_map"]), spec, config).to(device)
        table = parameter_summary(model)
        optimizer = make_optimizer(model, spec, config)
        scaler = _make_scaler(config, device)
        before = {name: parameter.detach().cpu().clone() for name, parameter in model.named_parameters()}
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        model.train()
        batches = next(iter(_accumulation_groups(_loader(dataset, config, True), config["accumulation_steps"])))
        skipped = 0
        # FP16 초깃값 overflow는 같은 CPU 누적 묶음 전체를 재검사한다.
        for attempt in range(1, 9):
            losses, updated = _training_step(model, optimizer, scaler, batches, config)
            if updated:
                break
            skipped += 1
            print(f"AMP scale 조정 {attempt}/8: optimizer 갱신 전 재검사", flush=True)
        else:
            raise FloatingPointError("AMP scale 조정 후에도 갱신하지 못했습니다. loss·입력·학습률을 확인하세요.")
        if any(parameter.grad is not None and not torch.isfinite(parameter.grad).all()
               for parameter in model.parameters()):
            raise FloatingPointError("한 배치 검사에서 gradient가 유한하지 않습니다. 입력·학습률을 확인하세요.")
        changed, frozen_ok = False, True
        for name, parameter in model.named_parameters():
            same = torch.equal(before[name], parameter.detach().cpu())
            if parameter.requires_grad:
                changed |= not same
            else:
                frozen_ok &= same
        if not changed or not frozen_ok or not optimizer.state:
            raise RuntimeError("실제 가중치 갱신 또는 고정 범위가 잘못됐습니다. 학습 설정을 확인하세요.")
        validation = make_dataset(bundle, "val")
        validation = PillDataset(validation.frame.head(1), validation.image_dir, validation.objects)
        measured = validation_loss(model, validation, config)
        records, _ = collect_predictions(model, validation, config, bundle["label_to_category"], True)
        converted = submission_frame(records[:1], bundle["category_to_label"])
        figure = preview_records(records[:1], count=1, score_threshold=config["preview_score"])
        try:
            figure.canvas.draw()
        finally:
            plt.close(figure)
        return {"passed": True, "signature": stable_hash({"recipe": _recipe(config, bundle), "spec": spec}),
                "loss": sum(float(value) for value in losses.values()), "validation_loss": measured["val_loss"],
                "batch_size": config["batch_size"], **_batch_policy(config),
                "micro_batches": len(batches), "processed_images": sum(len(images) for images, _ in batches),
                "precision": config["precision"],
                "optimizer_steps": 1, "amp_skipped_attempts": skipped, "amp_scale": scaler.get_scale(),
                "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1024**3 if device.type == "cuda" else None,
                "trainable_changed": changed, "frozen_unchanged": frozen_ok,
                "visualization_checked": True, "submission_rows": len(converted),
                "parameters": table.to_dict("records")}
    except torch.cuda.OutOfMemoryError:
        raise RuntimeError("현재 batch·정밀도로 GPU 메모리가 부족합니다. " +
                           f"현재 batch={config['batch_size']}, 누적={config['accumulation_steps']}입니다. " +
                           "다른 GPU 작업과 남은 메모리를 확인하고 셀 2 설정을 점검하세요.") from None
    finally:
        del model, optimizer, scaler, before
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


def _atomic_save(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as stream:
        temporary = Path(stream.name)
    torch.save(value, temporary)
    os.replace(temporary, path)


def _rng_state(generator):
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "loader": generator.get_state()}


def _restore_rng(state, generator):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"]:
        if len(state["cuda"]) != torch.cuda.device_count():
            raise RuntimeError("재개 시 GPU 구성이 다릅니다. 같은 구성에서 재개하세요.")
        torch.cuda.set_rng_state_all(state["cuda"])
    generator.set_state(state["loader"])


def completed_results(root, config, bundle):
    """이번 설계에서 완료된 실행만 읽는다. 과거 베이스라인은 읽지 않는다.

    Args:
        root (str | Path): 새 실험 전용 결과 루트.
        config (dict): 공통 설정.
        bundle (dict): 데이터 묶음.

    Returns:
        DataFrame: 완료 실행 표.
    """
    folder = experiment_root(root, config, bundle)
    rows = [json.loads(path.read_text()) for path in sorted(folder.glob("*/result.json"))]
    columns = ["candidate_id", "spec_id", "mode", "lr", "seed", "best_val_map_75_95", "best_epoch",
               "last_epoch", "precision", "micro_batch_size", "accumulation_steps", "effective_batch_size", "training_seconds", "inference_ms", "peak_gpu_gb", "run_name", "run_dir"]
    return pd.DataFrame(rows).reindex(columns=columns)





def run_experiment(bundle, config, spec, root, smoke_report=None, resume_dir=None):
    """필수 스모크 후 독립 학습 또는 명시적 체크포인트 재개를 수행한다.

    Args:
        bundle (dict): 고정 데이터.
        config (dict): 공통 설정.
        spec (dict): 학습 방식·학습률·seed.
        root (str | Path): 새 실험 결과 루트.
        smoke_report (dict | None): 일치하는 설정의 스모크 증명. 없거나 다르면 다시 검사.
        resume_dir (str | Path | None): 재개할 실행 폴더. 자동 재개하지 않음.

    Returns:
        dict: 완료 실행 결과.
    """
    recipe = _recipe(config, bundle)
    signature = stable_hash({"recipe": recipe, "spec": spec})
    folder = experiment_root(root, config, bundle)
    existing = completed_results(root, config, bundle)
    match = existing.loc[existing["spec_id"].eq(spec["spec_id"])]
    if resume_dir is None and not match.empty:
        if len(match) != 1:
            raise ValueError("같은 조건 결과가 여러 개입니다. 결과 목록에서 사용할 run을 명시하세요.")
        print("완료된 이번 실험 결과 재사용:", match.iloc[0]["run_name"])
        return match.iloc[0].to_dict()
    if not smoke_report or not smoke_report.get("passed") or smoke_report.get("signature") != signature:
        print("본 학습 설정으로 스모크 테스트:", spec["spec_id"], flush=True)
        smoke_report = smoke_test(bundle, config, spec)
    device = _device(config)
    seed_everything(spec["seed"], config["deterministic"])
    identifier = uuid.uuid4().hex[:12]
    name = run_name(config, spec)
    run_dir = Path(resume_dir) if resume_dir is not None else folder / f"{name}__{identifier}"
    if resume_dir is None:
        run_dir.mkdir(parents=True, exist_ok=False)
        manifest = {"signature": signature, "recipe": recipe, "spec": spec, "run_id": identifier,
                    "run_name": name, "run_dir": str(run_dir.resolve()),
                    "batch_policy": _batch_policy(config)}
        write_json(run_dir / "config.json", manifest)
        bundle["class_map"].to_csv(run_dir / "class_map.csv", index=False)
    else:
        manifest = json.loads((run_dir / "config.json").read_text())
        if manifest["signature"] != signature or (run_dir / "result.json").exists():
            raise ValueError("재개 설정이 다르거나 이미 완료된 실행입니다. 재개할 폴더를 확인하세요.")
    write_json(run_dir / "smoke.json", smoke_report)
    generator = torch.Generator().manual_seed(spec["seed"])
    train_loader = _loader(make_dataset(bundle, "train"), config, True, generator)
    model = build_model(len(bundle["class_map"]), spec, config, load_coco=resume_dir is None).to(device)
    optimizer = make_optimizer(model, spec, config)
    updates_per_epoch = math.ceil(len(train_loader) / config["accumulation_steps"])
    scheduler = make_scheduler(optimizer, updates_per_epoch, config)
    scaler = _make_scaler(config, device)
    validation_dataset = make_dataset(bundle, "val")
    best, best_epoch, stale, start_epoch, history, previous_seconds = -1.0, 0, 0, 1, [], 0.0
    resume_state = None
    if resume_dir is not None:
        resume_state = torch.load(run_dir / "last.pt", map_location="cpu", weights_only=False)
        if resume_state["signature"] != signature:
            raise ValueError("체크포인트 설정이 다릅니다. 같은 실험 폴더를 지정하세요.")
        model.load_state_dict(resume_state["model"], strict=True)
        optimizer.load_state_dict(resume_state["optimizer"])
        scheduler.load_state_dict(resume_state["scheduler"])
        if scaler.is_enabled() and "scaler" not in resume_state:
            raise ValueError("AMP scaler 상태가 없는 체크포인트입니다. 같은 AMP 실행의 last.pt를 지정하세요.")
        scaler.load_state_dict(resume_state.get("scaler", {}))
        best, best_epoch, stale = resume_state["best"], resume_state["best_epoch"], resume_state["stale"]
        start_epoch, history = resume_state["epoch"]+1, resume_state["history"]
        previous_seconds = resume_state["training_seconds"]
        pd.DataFrame(history).to_csv(run_dir / "metrics.csv", index=False)
    wandb_run = None
    try:
        if config["use_wandb"]:
            import wandb
            wandb_run = wandb.init(project=config["wandb_project"], entity=config["wandb_entity"],
                                   name=name, id=manifest["run_id"], resume="must" if resume_dir else "never",
                                   mode=config["wandb_mode"], dir=str(run_dir), config=manifest,
                                   group=folder.name, tags=["cj", "faster", spec["mode"], config["experiment_tag"]])
        if resume_state is not None:
            _restore_rng(resume_state["rng"], generator)
        else:
            # 로깅 초기화가 학습 난수에 영향을 주지 않게 다시 설정한다.
            seed_everything(spec["seed"], config["deterministic"])
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        training_start = time.perf_counter()
        peak = float(resume_state.get("peak_gpu_gb", 0)) if resume_state else 0.0
        # 종료 직후 result.json 저장 전에 중단된 실행은 반복 학습하지 않는다.
        epochs = range(start_epoch, config["max_epochs"]+1) if stale < config["patience"] else []
        for epoch in epochs:
            epoch_start = time.perf_counter()
            model.train()
            losses_total, seen = {}, 0
            optimizer_steps, skipped_steps = 0, 0
            first_lr = optimizer.param_groups[0]["lr"]
            micro_batches = 0
            for batches in _accumulation_groups(train_loader, config["accumulation_steps"]):
                group_images = sum(len(images) for images, _ in batches)
                loss_dict, updated = _training_step(model, optimizer, scaler, batches, config)
                if updated:
                    optimizer_steps += 1
                    scheduler.step()
                else:
                    skipped_steps += 1
                for key, value in loss_dict.items():
                    losses_total[key] = losses_total.get(key, 0.0) + value * group_images
                seen += group_images
                micro_batches += len(batches)
            # epoch 경계에는 미처리 gradient가 없다. 검증 전에 메모리를 돌려준다.
            optimizer.zero_grad(set_to_none=True)
            if seen == 0 or optimizer_steps == 0:
                raise RuntimeError("이 epoch에서 가중치를 갱신하지 못했습니다. AMP scale·loss·데이터를 확인하세요.")
            measured_loss = validation_loss(model, validation_dataset, config)
            records, speed = collect_predictions(model, validation_dataset, config,
                                                  bundle["label_to_category"], True)
            score = evaluate_records(records, detailed=False)["summary"]["map_75_95"]
            if not math.isfinite(score) or score < 0:
                raise RuntimeError("validation mAP를 계산할 수 없습니다. 정답·클래스 연결을 확인하세요.")
            improved = score > best
            if improved:
                best, best_epoch, stale = score, epoch, 0
            else:
                stale += 1
            peak = max(peak, torch.cuda.max_memory_allocated()/1024**3 if device.type == "cuda" else 0.0)
            row = {"epoch": epoch, "train_loss": sum(losses_total.values())/seen,
                   "val_map_75_95": score, "lr": first_lr, "last_lr": optimizer.param_groups[0]["lr"],
                   "epoch_seconds": time.perf_counter()-epoch_start, "val_inference_ms": speed["model_ms_per_image"],
                   "peak_gpu_gb": peak, "amp_scale": scaler.get_scale(),
                   "optimizer_steps": optimizer_steps, "amp_skipped_steps": skipped_steps,
                   "micro_batches": micro_batches, "train_images": seen,
                   "optimizer_groups": optimizer_steps + skipped_steps,
                   "last_group_images": group_images, **_batch_policy(config),
                   **measured_loss, **{key: value/seen for key, value in losses_total.items()}}
            history.append(row)
            if wandb_run is not None:
                try:

                    import wandb
                    previews = []
                    for record in records[:config["preview_count"]]:
                        truth, pred = record["target"], record["prediction"]
                        image = draw_boxes(record["image_path"], [xyxy_to_xywh(b.tolist()) for b in truth["boxes"]],
                                           [f"GT:{int(c)}" for c in truth["labels"]], color="lime")
                        kept = torch.where(pred["scores"] >= config["preview_score"])[0][:20]
                        image = draw_boxes(image, [xyxy_to_xywh(pred["boxes"][i].tolist()) for i in kept],
                                           [f"P:{int(pred['labels'][i])} {float(pred['scores'][i]):.2f}" for i in kept], color="red")
                        image.thumbnail((1000, 1000))
                        previews.append(wandb.Image(image, caption=f"epoch {epoch} | ID {record['image_id']} | green=GT, red=prediction"))
                    wandb_run.log({**row, "Media/validation": previews}, step=epoch)
                except Exception as error:
                    print(f"W&B 이미지·기록 실패: {type(error).__name__}: {error}. 체크포인트와 로컬 지표는 계속 저장합니다.", flush=True)
            elapsed = previous_seconds+time.perf_counter()-training_start
            state = {"signature": signature, "epoch": epoch, "model": model.state_dict(),
                     "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                     "scaler": scaler.state_dict(), "precision": config["precision"],
                     "batch_policy": _batch_policy(config),
                     "best": best, "best_epoch": best_epoch, "stale": stale, "history": history,
                     "training_seconds": elapsed, "peak_gpu_gb": peak, "rng": _rng_state(generator)}
            if improved:
                _atomic_save(run_dir / "best.pt", {"signature": signature, "epoch": epoch,
                                                    "model": model.state_dict(), "map_75_95": score})
            _atomic_save(run_dir / "last.pt", state)
            pd.DataFrame(history).to_csv(run_dir / "metrics.csv", index=False)
            print(f"{spec['spec_id']} | {epoch:03d}/{config['max_epochs']} | loss={row['train_loss']:.4f} "
                  f"| val_loss={row['val_loss']:.4f} | mAP={score:.4f} | patience={stale}/{config['patience']}", flush=True)
            if stale >= config["patience"]:
                break
        if not history:
            raise RuntimeError("학습 이력이 없습니다. 체크포인트와 epoch 설정을 확인하세요.")
        best_row = next(row for row in history if row["epoch"] == best_epoch)
        result = {**spec, "best_val_map_75_95": best, "best_epoch": best_epoch,
                  "last_epoch": history[-1]["epoch"], "training_seconds": previous_seconds+time.perf_counter()-training_start,
                  "inference_ms": best_row["val_inference_ms"], "peak_gpu_gb": peak,
                  "precision": config["precision"], **_batch_policy(config),
                  "run_name": name, "run_dir": str(run_dir.resolve())}
        write_json(run_dir / "result.json", result)
        return result
    finally:
        if wandb_run is not None:
            wandb_run.finish()
        del model, optimizer, scheduler, scaler
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


def predict_run(run_dir, dataset, bundle, config, with_targets=True, force=False):
    """저장된 best 모델의 추론을 실행하거나 동일 조건 캐시를 재사용한다.

    Args:
        run_dir (str | Path): 이번 설계의 실행 폴더.
        dataset (PillDataset): validation 또는 Test Dataset.
        bundle (dict): 학습 데이터·클래스 매핑.
        config (dict): 공통 설정.
        with_targets (bool): validation 정답 포함 여부.
        force (bool): 캐시가 일치해도 추론 재실행 여부.

    Returns:
        dict: records·추론 속도·캐시 식별자.
    """
    run_dir = Path(run_dir)
    manifest = json.loads((run_dir / "config.json").read_text())
    spec = manifest["spec"]
    if manifest["signature"] != stable_hash({"recipe": _recipe(config, bundle), "spec": spec}):
        raise ValueError("분석 대상의 코드·설정·데이터가 다릅니다. 해당 실행과 같은 환경을 준비하세요.")
    checkpoint = run_dir / "best.pt"
    image_identity = [(r.file_name, file_sha256(dataset.image_dir / r.file_name)) for r in dataset.frame.itertuples()]
    identity = stable_hash({"checkpoint": file_sha256(checkpoint), "images": image_identity,
                            "frame": dataset.frame[["file_name", "image_id", "width", "height"]].to_dict("records"),
                            "with_targets": with_targets, "signature": manifest["signature"]})
    path = run_dir / ("validation_predictions.pt" if with_targets else "test_predictions.pt")
    if path.exists() and not force:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        if payload["identity"] == identity:
            # 캐시 생성 후 저장 위치가 바뀌어도 현재 이미지 경로를 사용한다.
            for record in payload["records"]:
                record["image_path"] = str(dataset.image_dir / record["file_name"])
            return payload
    model = build_model(len(bundle["class_map"]), spec, config, load_coco=False)
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    if state["signature"] != manifest["signature"]:
        raise ValueError("best.pt와 실험 설정이 다릅니다. 같은 run의 파일을 지정하세요.")
    model.load_state_dict(state["model"], strict=True)
    model.to(_device(config))
    try:
        records, speed = collect_predictions(model, dataset, config, bundle["label_to_category"], with_targets)
        payload = {"identity": identity, "records": records, "speed": speed}
        _atomic_save(path, payload)
        return payload
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def export_comparison_run(run_dir, dataset, bundle, config, predictor=None):
    """비교 노트북이 재학습 없이 읽을 validation 예측과 검사 정보를 저장한다.

    Args:
        run_dir (str | Path): 완료된 학습 폴더.
        dataset (PillDataset): 전체 validation Dataset.
        bundle (dict): 고정 데이터와 클래스 매핑.
        config (dict): 해당 학습 설정.
        predictor (callable | None): Cascade 전용 환경의 추론 함수. None이면 현재 모델 추론.

    Returns:
        dict: 비교 입력 파일과 데이터·환경·평가 정책.
    """
    directory = Path(run_dir)
    inference = predict_run if predictor is None else predictor
    payload = inference(directory, dataset, bundle, config, with_targets=True)
    config_doc = json.loads((directory / "config.json").read_text())
    result = json.loads((directory / "result.json").read_text())
    if len(payload["records"]) != len(dataset):
        raise ValueError("전체 validation 예측이 아닙니다. 셀 7의 추론을 다시 실행하세요.")
    # CSV 저장·복원 후에도 같은 mAP인지 확인한다.
    checked = verify_roundtrip(payload["records"], bundle["class_map"]["category_id"],
                               directory / "analysis/validation_roundtrip.csv")
    prediction_path = directory / "validation_predictions.pt"
    document = {
        "format": "pill_detection_notebook_comparison_v1",
        "data_identity": bundle["identity"], "common_code": common_code_identity(),
        "class_map": bundle["class_map"].to_dict("records"),
        "iou_thresholds": IOU_THRESHOLDS, "max_detections": config["max_detections"],
        "score_threshold": config["score_threshold"],
        "prediction_file": prediction_path.name, "prediction_sha256": file_sha256(prediction_path),
        "run_signature": config_doc["signature"], "spec": config_doc["spec"],
        "runtime": config_doc["recipe"]["runtime"], "config": config_doc["recipe"]["config"],
        "speed": payload["speed"], "roundtrip": checked,
        "model": config_doc["recipe"]["runtime"]["model"],
        "result": {key: value for key, value in result.items() if key != "run_dir"},
    }
    write_json(directory / "comparison_manifest.json", document)
    return {"run_dir": str(directory), "seed": document["spec"]["seed"],
            "manifest": str(directory / "comparison_manifest.json"), "map_75_95": checked["after"]}


def _make_scaler(config, device):
    """CUDA AMP 설정에 맞는 손실 스케일러를 만든다.

    Args:
        config (dict): 정밀도 설정.
        device (torch.device): 학습 장치.

    Returns:
        GradScaler: FP32에서는 비활성화한 스케일러.
    """
    enabled = config["precision"] == "amp_fp16"
    if enabled and device.type != "cuda":
        raise RuntimeError("AMP FP16 학습에는 CUDA가 필요합니다. Kaggle GPU 커널을 선택하세요.")
    return torch.amp.GradScaler("cuda", enabled=enabled)


def _batch_policy(config):
    """실제 배치와 누적 후 목표 배치를 구분해 기록한다.

    Args:
        config (dict): 실제 배치·누적 횟수 설정.

    Returns:
        dict: 단일 GPU 기준 실제 배치·누적 횟수·목표 실효 배치.
    """
    return {"micro_batch_size": config["batch_size"],
            "accumulation_steps": config["accumulation_steps"],
            "effective_batch_size": config["batch_size"] * config["accumulation_steps"]}


def _accumulation_groups(loader, steps):
    """CPU의 작은 배치를 묶으며 마지막 불완전한 묶음도 반환한다.

    Args:
        loader (DataLoader): 원본 CPU 이미지와 정답을 읽는 로더.
        steps (int): 가중치 갱신 전 누적할 최대 작은 배치 수.

    Yields:
        list: GPU로 한 번에 올리지 않을 작은 배치 목록.
    """
    group = []
    for batch in loader:
        group.append(batch)
        if len(group) == steps:
            yield group
            group = []
    if group:
        yield group


def _training_step(model, optimizer, scaler, batches, config):
    """작은 배치별 역전파 후 누적 묶음마다 한 번만 가중치를 갱신한다.

    Args:
        model (Module): 학습 모드의 검출 모델.
        optimizer (Optimizer): 학습 optimizer.
        scaler (GradScaler): 묶음 전체에서 같은 scale을 유지할 스케일러.
        batches (list): CPU 이미지·정답의 작은 배치 목록.
        config (dict): 학습 정밀도·누적 횟수 설정.

    Returns:
        tuple: 실제 이미지 수로 가중 평균한 세부 loss와 갱신 성공 여부.
    """
    if not batches or len(batches) > config["accumulation_steps"]:
        raise ValueError("누적 묶음 크기가 잘못됐습니다. accumulation_steps와 로더를 확인하세요.")
    total_images = sum(len(images) for images, _ in batches)
    if total_images == 0:
        raise ValueError("빈 학습 묶음입니다. 고정 학습 분할을 확인하세요.")
    device = next(model.parameters()).device
    optimizer.zero_grad(set_to_none=True)
    means = {}
    for images, targets in batches:
        if not images or len(images) != len(targets) or len(images) > config["batch_size"]:
            raise ValueError("작은 배치의 이미지·정답 수가 잘못됐습니다. batch_size와 로더를 확인하세요.")
        inputs = [image.to(device) for image in images]
        truth = [{key: value.to(device) for key, value in target.items()} for target in targets]
        weight = len(images) / total_images
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=config["precision"] == "amp_fp16"):
            losses = model(inputs, truth)
            # 마지막 묶음이 16개 미만이어도 실제 이미지 수로 평균한다.
            # 검출 loss의 내부 정규화·RoI 샘플링 때문에 물리 배치 16과 완전히 같지는 않다.
            loss = sum(losses.values()) * weight
        if not torch.isfinite(loss):
            raise FloatingPointError("학습 loss가 유한하지 않습니다. 이미지·라벨·학습률을 확인하세요.")
        scaler.scale(loss).backward()
        for key, value in losses.items():
            means[key] = means.get(key, 0.0) + float(value.detach()) * weight
        # 이전 작은 배치의 GPU 입력과 그래프를 다음 forward까지 붙잡지 않는다.
        del inputs, truth, losses, loss
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    updated = not scaler.is_enabled() or scaler.get_scale() >= old_scale
    return means, updated


@contextlib.contextmanager
def _validation_loss_context(model, seed):
    """검출 loss 경로만 켜고 모드·난수·정규화 버퍼를 보존한다.

    Args:
        model (Module): 현재 epoch의 모델.
        seed (int): RPN·RoI 표본 추출에 사용할 고정 seed.
    """
    modes = {module: module.training for module in model.modules()}
    buffers = {name: value.detach().clone() for name, value in model.named_buffers()}
    python_state, numpy_state = random.getstate(), np.random.get_state()
    device = next(model.parameters()).device
    devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == "cuda" else []
    try:
        with torch.random.fork_rng(devices=devices):
            random.seed(seed)
            np.random.seed(seed)
            torch.random.default_generator.manual_seed(seed)
            if device.type == "cuda":
                with torch.cuda.device(device):
                    torch.cuda.manual_seed(seed)
            model.train()
            for module in model.modules():
                if isinstance(module, (torch.nn.modules.batchnorm._BatchNorm,
                                       torch.nn.modules.dropout._DropoutNd)):
                    module.eval()
            with torch.no_grad(), torch.autocast(device_type=device.type, enabled=False):
                yield
    finally:
        changed = False
        with torch.no_grad():
            for name, value in model.named_buffers():
                if not torch.equal(value, buffers[name]):
                    changed = True
                    value.copy_(buffers[name])
        for module, training in modes.items():
            module.training = training
        random.setstate(python_state)
        np.random.set_state(numpy_state)
        if changed:
            raise RuntimeError("validation loss 측정 중 버퍼가 바뀌어 복구했습니다. 모델 정규화 설정을 확인하세요.")


def validation_loss(model, dataset, config):
    """가중치 갱신 없이 validation 전체의 FP32 loss를 측정한다.

    Args:
        model (Module): 현재 epoch의 학습 모델.
        dataset (Dataset): 고정 validation 데이터.
        config (dict): 측정 batch와 고정 난수 설정.

    Returns:
        dict: 이미지 평균 전체 loss와 네 세부 loss.
    """
    device = next(model.parameters()).device
    loader = DataLoader(dataset, batch_size=config["val_loss_batch_size"], shuffle=False,
                        num_workers=config["num_workers"], collate_fn=collate_detection,
                        generator=torch.Generator().manual_seed(config["loss_measurement_seed"]),
                        pin_memory=device.type == "cuda")
    totals, seen = {}, 0
    expected = {"loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"}
    with _validation_loss_context(model, config["loss_measurement_seed"]):
        for images, targets in loader:
            values = model([image.to(device) for image in images],
                           [{key: value.to(device) for key, value in target.items()} for target in targets])
            if set(values) != expected:
                raise RuntimeError("검출 loss 네 종류가 필요합니다. Faster R-CNN 모델 구성을 확인하세요.")
            for key, value in values.items():
                if value.numel() != 1 or not torch.isfinite(value):
                    raise FloatingPointError("validation loss가 유한하지 않습니다. 라벨·모델을 확인하세요.")
                totals[key] = totals.get(key, 0.0) + float(value.detach()) * len(images)
            seen += len(images)
    if seen != len(dataset) or seen == 0:
        raise RuntimeError("validation 전체를 측정하지 못했습니다. 고정 분할을 확인하세요.")
    components = {f"val_{key}": value / seen for key, value in totals.items()}
    return {"val_loss": sum(components.values()), **components}


def training_plan(root, config, bundle, seeds, resume_dirs):
    """완료·미완료 실행을 표시하고 명시적인 재개 경로를 점검한다.

    Args:
        root (Path): 학습 결과 루트.
        config (dict): 고정 학습 설정.
        bundle (dict): 고정 데이터 묶음.
        seeds (list): 실행할 seed.
        resume_dirs (dict): seed별 명시적인 재개 폴더.

    Returns:
        DataFrame: seed별 예정 작업과 발견한 미완료 폴더.
    """
    if not seeds or len(set(seeds)) != len(seeds) or any(seed not in (0, 1, 2) for seed in seeds):
        raise ValueError("TRAIN_SEEDS에는 중복 없이 0·1·2 중 실행할 seed를 지정하세요.")
    if not set(resume_dirs).issubset(set(seeds)):
        raise ValueError("RESUME_DIRS의 seed는 TRAIN_SEEDS에도 있어야 합니다.")
    completed = completed_results(root, config, bundle)
    folder = experiment_root(root, config, bundle)
    rows = []
    for seed in seeds:
        spec = experiment_spec("ft", config["learning_rate"], seed)
        matches = completed.loc[completed["spec_id"].eq(spec["spec_id"])]
        if len(matches) > 1:
            raise ValueError("동일 seed의 완료 결과가 중복됩니다. 결과 폴더를 확인하세요.")
        partial = []
        signature = stable_hash({"recipe": _recipe(config, bundle), "spec": spec})
        for path in sorted(folder.glob("*/config.json")):
            manifest = json.loads(path.read_text())
            if manifest.get("signature") == signature and not (path.parent / "result.json").exists():
                partial.append(str(path.parent.resolve()))
        resume = resume_dirs.get(seed)
        if not matches.empty and resume is not None:
            raise ValueError("완료된 seed에는 재개 경로를 지정할 수 없습니다. RESUME_DIRS를 확인하세요.")
        if resume is not None:
            directory = Path(resume).expanduser().resolve()
            if not (directory / "last.pt").is_file() or not (directory / "config.json").is_file():
                raise ValueError("재개에는 같은 실행 폴더의 last.pt와 config.json이 필요합니다.")
            manifest = json.loads((directory / "config.json").read_text())
            if manifest.get("signature") != signature:
                raise ValueError("재개 실행의 코드·설정·데이터·환경이 다릅니다. 같은 조건의 실행을 지정하세요.")
            if (directory / "result.json").exists():
                raise ValueError("이미 완료된 실행입니다. RESUME_DIRS에서 이 seed를 제거하세요.")
        action = "완료 결과 재사용" if not matches.empty else ("명시적 재개" if resume else "COCO에서 새 학습")
        rows.append({"seed": seed, "mode": "ft", "lr": config["learning_rate"],
                     "precision": config["precision"], **_batch_policy(config), "run_name": run_name(config, spec),
                     "action": action, "resume_dir": str(resume) if resume else None,
                     "unfinished_runs": partial})
    plan = pd.DataFrame(rows)
    # pandas 3의 문자열 추론이 None을 NaN으로 바꾸지 않게 유지한다.
    plan["resume_dir"] = pd.Series([row["resume_dir"] for row in rows], dtype=object)
    return plan


def plot_run_history(run_dirs):
    """학습 loss·validation loss·validation mAP를 별도 축에 그린다.

    Args:
        run_dirs (list): metrics.csv를 포함한 실행 폴더.

    Returns:
        Figure: 측정 batch와 정밀도를 구분한 세 학습 곡선.
    """
    import matplotlib.pyplot as plt
    figure, axes = plt.subplots(1, 3, figsize=(17, 4))
    for directory in run_dirs:
        directory = Path(directory)
        history = pd.read_csv(directory / "metrics.csv")
        manifest = json.loads((directory / "config.json").read_text())
        label = f"seed {manifest['spec']['seed']}"
        for axis, key in zip(axes, ("train_loss", "val_loss", "val_map_75_95")):
            if key in history:
                axis.plot(history["epoch"], history[key], label=label)
        best = int(history.loc[history["val_map_75_95"].idxmax(), "epoch"])
        axes[2].axvline(best, linestyle=":", alpha=0.4)
    titles = ["Train loss (training batch / training precision)",
              "Validation loss (fixed batch / FP32)", "Validation bbox mAP@[.75:.05:.95]"]
    for axis, title in zip(axes, titles):
        axis.set(title=title, xlabel="Epoch")
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend()
    figure.tight_layout()
    return figure


def flatten_settings(value, prefix=""):
    """입력 설정과 실행 환경을 CSV의 key/value 행으로 펼친다.

    Args:
        value (object): JSON으로 저장한 설정.
        prefix (str): 상위 설정 이름.

    Returns:
        list: key/value 행.
    """
    if isinstance(value, dict):
        rows = []
        for key, child in value.items():
            rows.extend(flatten_settings(child, f"{prefix}.{key}" if prefix else str(key)))
        return rows
    if isinstance(value, (list, tuple)):
        return [{"key": prefix, "value": json.dumps(value, ensure_ascii=False)}]
    return [{"key": prefix, "value": value}]


_register_notebook_block("training")


## 2. 설정·데이터


### 셀 2 · 학습 설정·다운로드된 데이터·결과 경로

실제 batch 4, 누적 4회, 목표 실효 batch 16이다. TRAIN_SEEDS의 순서대로 seed 0 → 1 → 2를 학습한다.


In [ ]:
CONFIG = {
    **DEFAULT_CONFIG,
    "max_epochs": 100,
    "patience": 30,
    "image_size": 640,
    "batch_size": 2,           # GPU에서 한 번에 처리하는 이미지 수
    "accumulation_steps": 8,  # 4 × 4 = 목표 실효 배치 16
    "learning_rate": 3e-4,
    "precision": "amp_fp16",
    "experiment_tag": "fix_amp_acc8",
    "wandb_project": "pill-object-detection",
    "wandb_entity": "codeit-prj1-team2",
    "use_wandb": True,
    "wandb_mode": "online",
    "num_workers": 0,
    "val_loss_batch_size": 1,
    "loss_measurement_seed": 0,
}
TRAIN_SEEDS = [0, 1, 2]
# 같은 코드·AMP·배치 4·누적 4 조건의 last.pt만 재개한다. 이전 조건은 새 학습과 구분한다.
# 예: RESUME_DIRS = {1: "/kaggle/working/runs/cj_faster_experiments/.../실행폴더"}
RESUME_DIRS = {}

DATA_ROOT = Path("/kaggle/working/pill-training-data/data") if IS_KAGGLE else REPO_ROOT / "data"
IMAGE_DIR = DATA_ROOT / "raw/train_images"
ANNOTATION_DIR = DATA_ROOT / "raw/train_annotations"
SPLIT_PATH = DATA_ROOT / "processed/split.csv"
FIXES_PATH = DATA_ROOT / "processed/label_fixes.csv"
work_root = Path("/kaggle/working") if IS_KAGGLE else REPO_ROOT
OUTPUT_ROOT = work_root / "runs/cj_faster_experiments"
SUBMISSION_ROOT = work_root / "submissions/cj_faster_experiments"
TEST_IMAGE_DIR = DATA_ROOT / "raw/test_images"

validate_config(CONFIG)
paths = {"images": IMAGE_DIR, "annotations": ANNOTATION_DIR, "split": SPLIT_PATH, "fixes": FIXES_PATH}
display(pd.DataFrame([{"item": key, "path": str(path), "exists": path.exists()} for key, path in paths.items()]))
if not all(path.exists() for path in paths.values()):
    raise FileNotFoundError("필요한 데이터가 없습니다. Drive 복사 완료 여부와 DATA_ROOT를 확인하세요.")
if any(DATA_ROOT.rglob("*.part")):
    raise RuntimeError("다운로드 중인 .part 파일이 있습니다. 복사를 완료한 뒤 이 셀을 다시 실행하세요.")
print(f"학습: FT / 3e-4 | 실제 batch {CONFIG['batch_size']} × 누적 {CONFIG['accumulation_steps']} "
      f"= 목표 실효 batch {CONFIG['batch_size'] * CONFIG['accumulation_steps']} | AMP FP16 | seed:", TRAIN_SEEDS)
print("검증: bbox mAP@[0.75:0.05:0.95], FP32 추론·FP32 validation loss")
print("학습 결과:", OUTPUT_ROOT)
print("제출 CSV:", SUBMISSION_ROOT)
display(pd.DataFrame([{"setting": key, "value": str(value)} for key, value in CONFIG.items()]))


### 셀 3 · 고정 분할·수정표 적용·정답 시각화

이미지 내용과 라벨을 검증한다. 다운로드나 원본 파일 수정은 하지 않는다.


In [ ]:
print("고정 분할·수정 라벨·이미지 확인 중…", flush=True)
bundle = prepare_data(IMAGE_DIR, ANNOTATION_DIR, SPLIT_PATH, FIXES_PATH)
summary, support = data_summary(bundle)
train_dataset = make_dataset(bundle, "train")
val_dataset = make_dataset(bundle, "val")
RUN_GROUP_DIR = experiment_root(OUTPUT_ROOT, CONFIG, bundle)
RUN_GROUP_DIR.mkdir(parents=True, exist_ok=True)
write_json(RUN_GROUP_DIR / "settings.json", {
    "config": CONFIG, "environment": ENVIRONMENT, "data_identity": bundle["identity"],
    "train_seeds": TRAIN_SEEDS,
    "loss_measurement": {"validation": "FP32/no_grad/fixed_seed", "training": "online optimizer batches"},
})
bundle["class_map"].merge(support, on="category_id", how="left").to_csv(
    RUN_GROUP_DIR / "class_support.csv", index=False)
print("원본 bbox:", bundle["raw_boxes"], "| 수정 후:", bundle["fixed_boxes"], "| 수정:", bundle["fixes"])
print("데이터 식별자:", bundle["identity"][:12])
display(summary)
display(bundle["class_map"].merge(support, on="category_id", how="left"))
figure = plot_ground_truth(train_dataset, bundle["label_to_category"], count=3)
plt.show()
plt.close(figure)


## 3. GPU 확인·학습


### 셀 4 · 실행 계획·AMP 누적 갱신 확인

이 셀은 **1 epoch나 100 epoch 학습이 아니다.** batch 4를 네 번 역전파한 뒤 optimizer를 **한 번** 갱신한다.
같은 정밀도·누적으로 메모리·검증 loss·시각화·CSV를 확인하고 검사용 가중치는 버린다.
AMP 초기 overflow가 나면 같은 누적 묶음 전체를 최대 8회 재시도한다. 성공한 optimizer 갱신은 한 번이다.
본 학습은 COCO에서 독립적으로 시작한다. GPU 메모리가 부족해도 batch를 자동 변경하지 않는다.


In [ ]:
PLAN = training_plan(OUTPUT_ROOT, CONFIG, bundle, TRAIN_SEEDS, RESUME_DIRS)
display(PLAN)
SMOKE_REPORT = None
pending = PLAN.loc[PLAN["action"].ne("완료 결과 재사용")]
if pending.empty:
    print("선택한 seed는 모두 완료됐습니다. 셀 6부터 분석하세요.")
else:
    for row in pending.to_dict("records"):
        if row["unfinished_runs"] and row["resume_dir"] is None:
            print(f"seed {row['seed']}: 미완료 폴더가 있습니다. 이어가려면 셀 2의 RESUME_DIRS에 지정하세요.")
    SMOKE_SPEC = experiment_spec("ft", CONFIG["learning_rate"], int(pending.iloc[0]["seed"]))
    print("누적 4회·가중치 1회 갱신 확인 시작:", SMOKE_SPEC["spec_id"], flush=True)
    SMOKE_REPORT = smoke_test(bundle, CONFIG, SMOKE_SPEC)
    print({key: value for key, value in SMOKE_REPORT.items() if key != "parameters"})
    display(pd.DataFrame(SMOKE_REPORT["parameters"]))


### 셀 5 · W&B 인증·본 학습 실행 — 최대 100 epoch × seed 0·1·2

**이 셀을 실행하면 본 학습이 시작된다.** TRAIN_SEEDS 순서대로 진행한다.
batch 4를 최대 네 번 누적한 뒤 가중치를 한 번 갱신하며, 학습률 스케줄도 성공한 갱신에만 진행한다.
완료된 동일 조건은 재사용한다. 같은 코드·정밀도·실제 배치·누적 조건의 last.pt만 RESUME_DIRS로 재개한다.
API 키는 숨김 입력창 또는 WANDB_API_KEY로만 받는다. W&B 접근 오류는 entity·계정 권한을 확인한다.
매 epoch 완료 시 optimizer·scheduler·AMP scaler·난수 상태와 배치 정책을 last.pt에 저장한다.


In [ ]:
print("커널 연결 정상", flush=True)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/runs/cj_faster_experiments")
files = list(root.glob(
    "*/cj_faster_e100_s0_fix_amp_acc8_ft_lr3e-4__*/last.pt"
))

print("체크포인트 개수:", len(files), flush=True)
for path in files:
    print(path, flush=True)

In [ ]:
RESUME_DIRS = {0: str(files[0].parent)}
print("seed 0 재개 폴더:", RESUME_DIRS[0])

In [ ]:
from getpass import getpass

PLAN = training_plan(OUTPUT_ROOT, CONFIG, bundle, TRAIN_SEEDS, RESUME_DIRS)
display(PLAN)
pending = PLAN.loc[PLAN["action"].ne("완료 결과 재사용")]
if pending.empty:
    print("모든 선택 seed가 완료됐습니다. 재학습 없이 셀 6부터 진행하세요.")
else:
    for row in pending.to_dict("records"):
        checkpoints = [directory for directory in row["unfinished_runs"]
                       if (Path(directory) / "last.pt").is_file()]
        if checkpoints and row["resume_dir"] is None:
            raise RuntimeError(f"seed {row['seed']}의 재개 가능한 실행이 있습니다: {checkpoints}. "
                               "셀 2의 RESUME_DIRS에 폴더를 지정하고 셀 4부터 다시 실행하세요.")
    if not globals().get("SMOKE_REPORT") or not SMOKE_REPORT.get("passed"):
        raise RuntimeError("셀 4의 한 배치 검사를 먼저 통과하세요.")
    if CONFIG["use_wandb"] and CONFIG["wandb_mode"] == "online":
        import wandb
        wandb_key = os.environ.get("WANDB_API_KEY") or getpass("W&B API key: ")
        if not wandb_key.strip():
            del wandb_key
            raise RuntimeError("W&B API 키가 비었습니다. 숨김 입력창에 키를 입력하거나 셀 2에서 use_wandb를 False로 지정하세요.")
        try:
            wandb.login(key=wandb_key.strip(), verify=True)
        finally:
            del wandb_key
        print("W&B 저장 대상:", CONFIG["wandb_entity"], "/", CONFIG["wandb_project"])
    for row in pending.to_dict("records"):
        spec = experiment_spec("ft", CONFIG["learning_rate"], int(row["seed"]))
        print("\n본 학습 시작:", run_name(CONFIG, spec), flush=True)
        result = run_experiment(bundle, CONFIG, spec, OUTPUT_ROOT,
                                smoke_report=SMOKE_REPORT, resume_dir=row["resume_dir"])
        display(pd.DataFrame([result]))
results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
display(results)


In [ ]:
from pathlib import Path

root = Path("/kaggle/working/runs/cj_faster_experiments")
files = list(root.glob(
    "*/cj_faster_e100_s0_fix_amp_acc8_ft_lr3e-4__*/last.pt"
))

print("체크포인트 개수:", len(files), flush=True)
for path in files:
    print(path, flush=True)

## 4. 검증·오류 분석·공유


### 셀 6 · seed별 완료 결과·전체 설정 CSV

새 AMP 실험에서 완료한 seed만 집계한다. 기존 FP32 seed 0과의 3-seed 평균으로 표현하지 않는다.
settings_all.csv에 config·spec·환경·코드·데이터 식별자를 key/value로 저장한다.


In [ ]:
results = completed_results(OUTPUT_ROOT, CONFIG, bundle).sort_values("seed")
if results.empty:
    raise RuntimeError("완료된 실행이 없습니다. 셀 5 학습을 완료하거나 결과 경로·설정을 확인하세요.")
if results["spec_id"].duplicated().any():
    raise ValueError("같은 seed의 완료 결과가 중복됩니다. 사용할 결과 폴더를 확인하세요.")
results.to_csv(RUN_GROUP_DIR / "seed_results.csv", index=False)
setting_rows = []
for row in results.to_dict("records"):
    manifest = json.loads((Path(row["run_dir"]) / "config.json").read_text())
    setting_rows.extend({"run_name": row["run_name"], "seed": row["seed"], **setting}
                        for setting in flatten_settings(manifest))
pd.DataFrame(setting_rows).to_csv(RUN_GROUP_DIR / "settings_all.csv", index=False)
seed_summary = results.groupby(["candidate_id", "precision"]).agg(
    seed_count=("seed", "nunique"), map_mean=("best_val_map_75_95", "mean"),
    map_std=("best_val_map_75_95", "std"), training_seconds=("training_seconds", "mean"),
    inference_ms=("inference_ms", "mean"), peak_gpu_gb=("peak_gpu_gb", "max"),
)
seed_summary.to_csv(RUN_GROUP_DIR / "seed_summary.csv")
display(results)
display(seed_summary)
figure = plot_candidate_scores(results)
figure.savefig(RUN_GROUP_DIR / "seed_scores.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(figure)
print("집계 대상은 위에 표시된 AMP seed입니다. 기존 FP32 seed 0은 별도 실험입니다.")


### 셀 7 · 분석 대상 best 모델의 validation 전체 예측

ANALYSIS_RUN_DIR=None이면 이번 AMP 실험의 최고 validation mAP 실행을 선택한다.


In [ ]:
ANALYSIS_RUN_DIR = None
results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
if results.empty:
    raise RuntimeError("완료된 AMP 학습이 없습니다. 셀 5의 결과를 확인하세요.")
if ANALYSIS_RUN_DIR is None:
    ANALYSIS_RUN_DIR = Path(results.sort_values("best_val_map_75_95", ascending=False).iloc[0]["run_dir"])
else:
    ANALYSIS_RUN_DIR = Path(ANALYSIS_RUN_DIR)
validation_payload = predict_run(ANALYSIS_RUN_DIR, val_dataset, bundle, CONFIG, with_targets=True)
ANALYSIS_DIR = ANALYSIS_RUN_DIR / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print("분석 대상:", ANALYSIS_RUN_DIR)
print("추론 기록:", validation_payload["speed"])


### 셀 8 · mAP·클래스별/IoU별 AP·train/val loss 곡선


In [ ]:
report = evaluate_records(validation_payload["records"], bundle["class_map"], detailed=True)
display(pd.DataFrame([report["summary"]]))
display(report["classes"].sort_values("ap_75_95", na_position="last"))
display(report["ious"])
write_json(ANALYSIS_DIR / "summary.json", report["summary"])
report["classes"].to_csv(ANALYSIS_DIR / "class_ap.csv", index=False)
report["ious"].to_csv(ANALYSIS_DIR / "iou_ap.csv", index=False)
np.savez_compressed(ANALYSIS_DIR / "precision_recall.npz", precision=report["precision"],
                    category_ids=report["category_ids"])
results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
for filename, figure in [
    ("learning_curves.png", plot_run_history(results["run_dir"].tolist())),
    ("class_iou_ap.png", plot_ap(report)),
]:
    figure.savefig(ANALYSIS_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figure)
print("best·early stopping 기준은 val mAP입니다. train/val loss의 batch·정밀도 차이를 함께 확인하세요.")


### 셀 9 · 검출 혼동행렬·오류 유형·오류 이미지

행=정답, 열=예측. 마지막 행은 추가 예측, 마지막 열은 미검출이다.
IoU로 일대일 대응하며 다른 클래스로 예측한 경우도 연결한다.
IoU 0.75·score 0.30은 진단용 기준이며 mAP 계산용 예측은 score 0.001 정책을 유지한다.


In [ ]:
DIAGNOSTIC_IOU = 0.75
DIAGNOSTIC_SCORE = 0.30
if validation_payload is not None:
    matrix, errors = detection_errors(validation_payload["records"], bundle["class_map"]["category_id"],
                                       iou_threshold=DIAGNOSTIC_IOU, score_threshold=DIAGNOSTIC_SCORE)
    matrix.to_csv(ANALYSIS_DIR / "confusion_counts.csv")
    errors.to_csv(ANALYSIS_DIR / "error_events.csv", index=False)
    write_json(ANALYSIS_DIR / "diagnostic_settings.json", {"iou": DIAGNOSTIC_IOU, "score": DIAGNOSTIC_SCORE,
                                                          "rows": "true", "columns": "predicted"})
    display(errors["kind"].value_counts().rename_axis("kind").to_frame("events"))
    display(errors.loc[errors["kind"].ne("correct")].head(20))
    for filename, figure in [
        ("confusion_normalized.png", plot_confusion(matrix, normalize=True)),
        ("error_examples.png", preview_records(validation_payload["records"], errors, count=6,
                                               score_threshold=DIAGNOSTIC_SCORE)),
    ]:
        figure.savefig(ANALYSIS_DIR / filename, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(figure)


### 셀 10 · validation CSV 검증·두 모델 비교 자료 저장

이번 AMP 실험에서 완료한 seed별 자료를 저장한다. 기존 04 비교 노트북의 직접 비교에는 두 모델의 같은 조건·seed 0·1·2가 모두 필요하다. 현재 기본 AMP seed 1·2만으로는 해당 검사를 통과하지 않는다.


In [ ]:
results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
comparison_exports = [export_comparison_run(directory, val_dataset, bundle, CONFIG)
                      for directory in results["run_dir"]]
display(pd.DataFrame(comparison_exports))
pd.DataFrame(comparison_exports).to_csv(RUN_GROUP_DIR / "comparison_exports.csv", index=False)
if sorted(results["seed"].tolist()) == [0, 1, 2]:
    write_json(RUN_GROUP_DIR / "final_selection.json", {
        "candidate_id": "ft_lr3e-4", "basis": "validation_same_precision_seeds_0_1_2",
        "precision": CONFIG["precision"], "run_dirs": results.sort_values("seed")["run_dir"].tolist(),
    })
    print("같은 조건의 seed 0·1·2 비교 자료를 저장했습니다.")
else:
    print("완료 seed의 자료를 저장했습니다. 04의 직접 비교는 양쪽 모델의 동일 조건·seed 0·1·2가 필요합니다.")


## 5. 선택 · Test 예측·결과 보관


### 셀 11 · Test 전체 예측·Kaggle 제출 CSV

Test 이미지를 준비한 경우에만 실행한다. 없으면 안내만 표시한다.
빈 SUBMISSION_RUN_DIRS는 validation mAP가 가장 높은 AMP 실행 하나를 사용한다.
각 실행별 CSV를 만들며 seed 앙상블이나 Kaggle 자동 제출은 수행하지 않는다.


In [ ]:
SUBMISSION_RUN_DIRS = []
exports = []
if not TEST_IMAGE_DIR.is_dir():
    print("Test 이미지가 아직 없습니다. 필요할 때 TEST_IMAGE_DIR에 준비한 뒤 이 셀을 실행하세요.")
else:
    results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
    if not SUBMISSION_RUN_DIRS:
        if results.empty:
            raise RuntimeError("제출에 사용할 완료 학습이 없습니다. 셀 5 결과를 확인하세요.")
        SUBMISSION_RUN_DIRS = [results.sort_values("best_val_map_75_95", ascending=False).iloc[0]["run_dir"]]
    test_images = test_dataset(TEST_IMAGE_DIR)
    output_dir = SUBMISSION_ROOT / RUN_GROUP_DIR.name
    output_dir.mkdir(parents=True, exist_ok=True)
    test_images.frame.to_csv(output_dir / "test_manifest.csv", index=False)
    for directory in SUBMISSION_RUN_DIRS:
        directory = Path(directory)
        val_check = predict_run(directory, val_dataset, bundle, CONFIG, with_targets=True)
        verify_roundtrip(val_check["records"], bundle["class_map"]["category_id"],
                         directory / "analysis/validation_roundtrip.csv")
        payload = predict_run(directory, test_images, bundle, CONFIG, with_targets=False)
        if len(payload["records"]) != len(test_images):
            raise RuntimeError("Test 이미지 전체의 예측이 아닙니다. 해당 실행의 추론을 다시 수행하세요.")
        frame = submission_frame(payload["records"], bundle["class_map"]["category_id"])
        if frame.empty:
            raise RuntimeError("제출 예측이 비었습니다. 모델·score·Test 이미지를 확인하세요.")
        csv_path = output_dir / f"{directory.name}.csv"
        frame.to_csv(csv_path, index=False)
        loaded = pd.read_csv(csv_path)
        restored = restore_submission(loaded, payload["records"])
        submission_frame(restored, bundle["class_map"]["category_id"])
        if len(loaded) != len(frame):
            raise RuntimeError("저장한 CSV의 행 수가 다릅니다. 이 셀을 다시 실행하세요.")
        exports.append({"run": directory.name, "csv_path": str(csv_path),
                        "test_images": len(test_images), "prediction_rows": len(frame)})
    pd.DataFrame(exports).to_csv(output_dir / "submission_manifest.csv", index=False)
    display(pd.DataFrame(exports))
    print("csv_path의 예측 CSV를 Kaggle에 직접 제출하세요. test_manifest와 submission_manifest는 제출 파일이 아닙니다.")


### 셀 12 · 결과 ZIP 만들기

학습·분석이 끝나면 실행한다. 가중치를 포함하므로 파일이 클 수 있다. 환경·원본 데이터·W&B 인증 파일은 포함하지 않는다.


In [ ]:
import zipfile

archive_path = work_root / f"cj_faster_amp_results_{RUN_GROUP_DIR.name}.zip"
temporary = archive_path.with_suffix(".zip.part")
roots = [RUN_GROUP_DIR, SUBMISSION_ROOT / RUN_GROUP_DIR.name]
with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:
    for root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob("*")):
            relative = path.relative_to(root)
            if (not path.is_file() or path.is_symlink() or "wandb" in relative.parts
                    or path.name.startswith(".") or path.suffix == ".part"):
                continue
            archive.write(path, arcname=str(Path(root.parent.name) / root.name / relative))
temporary.replace(archive_path)
print("결과 ZIP:", archive_path)
print("크기(GB):", round(archive_path.stat().st_size / 1024**3, 3))
print("Kaggle 웹 노트북의 Output에서 ZIP을 다운로드해 보관하세요.")
